# FSOT Biohub Submission
**Original program port** (`fsot_rna_trinary_evolution_sim`):
zarr → vision detect → `fsot_core` + `fsot_cellular_bridge` linking → `submission.csv`

Default engine: `fsot` (peaks + FSOT math). Set `BIOHUB_ENGINE=fsot_unet` for U-Net gateway.

**Datasets:** fsot-offline-dependencies, fsot-nn-cellpose-weights, cellmot-baseline-artifacts, cellmot-ft-detector-biohub


In [ ]:
import os
import subprocess
import glob
import sys

# Force CPU even if Kaggle assigns a GPU worker (avoids P100/PyTorch hang + submission block).
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"
os.environ["OPENBLAS_NUM_THREADS"] = "4"

print("Kaggle input:", os.listdir("/kaggle/input"))

# Prefer fine-tuned Biohub detector weights, then baseline artifacts
_ft = glob.glob("/kaggle/input/**/cellmot-ft-detector-biohub/**/edge_predictor_best.pth", recursive=True)
_wt = _ft or glob.glob("/kaggle/input/**/edge_predictor_best.pth", recursive=True)
if _wt:
    os.environ["CELLMOT_UNET_WEIGHTS"] = _wt[0]
    print("U-Net weights:", _wt[0])
os.environ.setdefault("BIOHUB_ENGINE", "fsot_unet")
os.environ.setdefault("BIOHUB_DETECTOR", "peaks")
os.environ.setdefault("FSOT_LINK_MODE", "fsot_gate")
os.environ.setdefault("FSOT_GATE_FRAC", "0.48")
os.environ.setdefault("FSOT_GATE_ADAPTIVE", "0")
os.environ.setdefault("FSOT_GATE_RESCUE", "0")
os.environ.setdefault("CELLMOT_USE_FT", "1")
os.environ.setdefault("CELLMOT_DET_THRESHOLD", "0.99")
os.environ.setdefault("CELLMOT_EDGE_THRESHOLD", "0.3")
os.environ.setdefault("CELLMOT_USE_ILP", "1")
os.environ.setdefault("CELLMOT_ILP_MAX_EDGES", "35000")
os.environ.setdefault("CELLMOT_DET_TTA", "0")
os.environ.setdefault("CELLMOT_DEVICE", "cpu")
os.environ.setdefault("CELLMOT_NMS_UM", "8.0")
os.environ.setdefault("CELLMOT_POOL_UM", "8.0")
os.environ.setdefault("FSOT_GAP_LINK", "1")
os.environ.setdefault("KAGGLE_SUBMISSION_FAST_VALIDATE", "1")

_cellmot_wheels = glob.glob("/kaggle/input/**/cellmot-baseline-artifacts/**/wheels", recursive=True)
_offline_wheels = glob.glob("/kaggle/input/**/zarr-*.whl", recursive=True)
if _cellmot_wheels:
    _cmw = _cellmot_wheels[0]
    print(f"Cellmot wheels: {_cmw}")
    # Install every offline wheel except numpy/scipy (pip resolver upgrades numpy and breaks Kaggle).
    _skip = ("numpy-", "scipy-")
    for _whl in sorted(glob.glob(f"{_cmw}/*.whl")):
        _base = os.path.basename(_whl).lower()
        if _base.startswith(_skip):
            continue
        subprocess.run(
            f"pip install --no-index --no-deps --force-reinstall {_whl}",
            shell=True, capture_output=True, text=True,
        )
    import numpy as _np
    import polars as _pl
    import tracksdata as _td
    assert hasattr(_pl, "Float16"), f"polars {_pl.__version__} too old for tracksdata"
    print(f"numpy={_np.__version__} polars={_pl.__version__} tracksdata OK")
else:
    print("CRITICAL: cellmot-baseline-artifacts wheels not found")

if _offline_wheels:
    wheel_dir = os.path.dirname(_offline_wheels[0])
    print(f"Offline wheels: {wheel_dir}")
    _enable_deep = os.environ.get("FSOT_ENABLE_DEEP_DETECTOR", "0") == "1"
    if _enable_deep:
        import shutil as _sh
        _to_install = []
        for pat, dst in [
            (f"{wheel_dir}/torch-2.5*.whl", "/tmp/torch-2.5.1+cu121-cp312-cp312-linux_x86_64.whl"),
            (f"{wheel_dir}/torchvision-0.20*.whl", "/tmp/torchvision-0.20.1+cu121-cp312-cp312-linux_x86_64.whl"),
        ]:
            hits = sorted(glob.glob(pat))
            if hits:
                _sh.copy(hits[0], dst)
                _to_install.append(dst)
        if _to_install:
            subprocess.run(
                "pip install --no-index --no-deps --force-reinstall " + " ".join(_to_install),
                shell=True, capture_output=True, text=True,
            )
        cp = (
            f"pip install --no-index --no-deps --find-links {wheel_dir} "
            f"cellpose fastremap fill_voids imagecodecs roifile natsort tifffile segment_anything"
        )
        subprocess.run(cp, shell=True, capture_output=True, text=True)


In [ ]:
# Write cellmot_code_bundle.zip to the working dir (offline Kaggle bundle).
import base64
_B64 = "..."
with open('cellmot_code_bundle.zip', 'wb') as _f:
    _f.write(base64.b64decode(_B64))
print('cellmot_code_bundle.zip written to working dir.')

In [ ]:
import glob
import os
import sys
import zipfile

_candidates = []
if os.path.exists("/kaggle/working/cellmot_code_bundle.zip"):
    _candidates.append("/kaggle/working/cellmot_code_bundle.zip")
_candidates.extend(glob.glob("/kaggle/input/**/cellmot_code_bundle.zip", recursive=True))
for _bp in _candidates:
    with zipfile.ZipFile(_bp, "r") as _zf:
        _zf.extractall("/kaggle/working")
    sys.path.insert(0, "/kaggle/working/cellmot_bundle/src")
    sys.path.insert(0, "/kaggle/working/cellmot_bundle/scripts")
    print("cellmot_bundle extracted from", _bp)
    break
else:
    raise RuntimeError("cellmot_code_bundle.zip missing after embed step")


In [ ]:
# Write fsot_core.py to the working dir (offline Kaggle bundle).
import base64
_B64 = "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMw0KIiIiDQpGU09UIDIuMSBDT1JFIFNDQUxBUiBFTkdJTkUgIChmYXN0IGZsb2F0NjQgcG9ydCArIGVxdWl2YWxlbmNlIHZhbGlkYXRvcikNCj09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQ0KRmFpdGhmdWwgcG9ydCBvZiBEYW1pYW4gQXJ0aHVyIFBhbHVtYm8ncyBGU09UIHNjYWxhciBlbmdpbmUNCihgY29tcHV0ZV9TX0RfY2hhb3RpY2AgLyBgY29tcHV0ZV9zY2FsYXJgKSBmcm9tIHRoZSBMZWFuIDQgZm9ybWFsaXphdGlvbg0KKEZTT1QtMi4xLUxlYW46IEZTT1QvU2NhbGFyLmxlYW4pIGFuZCB0aGUgd29ya3NwYWNlIG1wbWF0aCByZWZlcmVuY2UNCihmc290X3JuYV90cmluYXJ5X2V2b2x1dGlvbl9zaW0ucHkgOjogY29tcHV0ZV9zY2FsYXIpLg0KDQpXSFkgVEhJUyBGSUxFIEVYSVNUUw0KLS0tLS0tLS0tLS0tLS0tLS0tLS0NClRoZSB2ZXJpZmllZCByZWZlcmVuY2UgZW5naW5lIHJ1bnMgYXQgbXAuZHBzID0gNTAgKDUwLWRpZ2l0IG1wbWF0aCkuIFRoYXQgaXMNCnRoZSBzb3VyY2Ugb2YgdHJ1dGgsIGJ1dCBmYXIgdG9vIHNsb3cgdG8gZXZhbHVhdGUgZm9yIHRob3VzYW5kcyBvZiBjZWxsLXBhaXJzDQphY3Jvc3MgbWFueSBmcmFtZXMgaW5zaWRlIEthZ2dsZSdzIDEyLWhvdXIgY2FwLiBUaGlzIG1vZHVsZSByZXByb2R1Y2VzIHRoZQ0KKnNhbWUqIGVuZ2luZSBpbiBmbG9hdDY0IHNvIGl0IHJ1bnMgYXQgY29tcGV0aXRpb24gc2NhbGUsIGFuZCBzaGlwcyBhDQp2YWxpZGF0b3IgKGB2YWxpZGF0ZV9hZ2FpbnN0X21wbWF0aGApIHRoYXQgcHJvdmVzIHRoZSBmbG9hdDY0IHJlc3VsdCBtYXRjaGVzDQp0aGUgbXBtYXRoIHJlZmVyZW5jZSB0byB+MWUtMTIgZm9yIHRoZSBiaW9sb2dpY2FsIGRvbWFpbiBhbmQgcGFyYW1ldGVyIHN3ZWVwcy4NCg0KTm90aGluZyBhYm91dCBGU09UIGlzIGFwcHJveGltYXRlZCBhd2F5IOKAlCBvbmx5IHRoZSBhcml0aG1ldGljIHByZWNpc2lvbiBvZiB0aGUNCm51bWVyaWMgZXZhbHVhdGlvbiBjaGFuZ2VzLCBhbmQgdGhhdCBjaGFuZ2UgaXMgbWVhc3VyZWQgYW5kIGJvdW5kZWQuDQoiIiINCg0KZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucw0KaW1wb3J0IG1hdGgNCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcw0KDQojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQ0KIyBGVU5EQU1FTlRBTCAmIERFUklWRUQgQ09OU1RBTlRTICAoZmxvYXQ2NCwgbWlycm9yIG9mIFNjYWxhci5sZWFuKQ0KIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0NClBJID0gbWF0aC5waQ0KRSA9IG1hdGguZQ0KUEhJID0gKDEuMCArIG1hdGguc3FydCg1LjApKSAvIDIuMA0KU1FSVDIgPSBtYXRoLnNxcnQoMi4wKQ0KR0FNTUFfRVVMRVIgPSAwLjU3NzIxNTY2NDkwMTUzMjkNCkNBVEFMQU5fRyA9IDAuOTE1OTY1NTk0MTc3MjE5MA0KDQpBTFBIQSA9IG1hdGgubG9nKFBJKSAvIChFICogUEhJKioxMykNClBTSV9DT04gPSAoRSAtIDEuMCkgLyBFICAgICAgICAgICAgICAgICAgICAgICMgPT0gMSAtIGV4cCgtMSkNCkVUQV9FRkYgPSAxLjAgLyAoUEkgLSAxLjApDQpCRVRBID0gMS4wIC8gbWF0aC5leHAoUEkqKlBJICsgKEUgLSAxLjApKQ0KR0FNTUFfQyA9IC1tYXRoLmxvZygyLjApIC8gUEhJDQpPTUVHQSA9IG1hdGguc2luKFBJIC8gRSkgKiBTUVJUMg0KVEhFVEFfUyA9IG1hdGguc2luKFBTSV9DT04gKiBFVEFfRUZGKQ0KUE9PRiA9IG1hdGguZXhwKCgtbWF0aC5sb2coUEkpIC8gRSkgLyAoRVRBX0VGRiAqIG1hdGgubG9nKFBISSkpKQ0KDQpDX0VGRiA9ICgxLjAgLSBQT09GICogbWF0aC5zaW4oVEhFVEFfUykpICogKDEuMCArIDAuMDEgKiBDQVRBTEFOX0cgLyAoUEkgKiBQSEkpKQ0KQV9CTEVFRCA9IG1hdGguc2luKFBJIC8gRSkgKiBQSEkgLyBTUVJUMg0KUF9WQVIgPSAtbWF0aC5jb3MoVEhFVEFfUyArIFBJKQ0KQl9JTiA9IENfRUZGICogKDEuMCAtIG1hdGguc2luKFRIRVRBX1MpIC8gUEhJKQ0KQV9JTiA9IEFfQkxFRUQgKiAoMS4wICsgbWF0aC5jb3MoVEhFVEFfUykgLyBQSEkpDQpTVUNUSU9OID0gUE9PRiAqICgtbWF0aC5jb3MoVEhFVEFfUyAtIFBJKSkNCkNIQU9TID0gR0FNTUFfQyAvIE9NRUdBDQpQX0JBU0UgPSBHQU1NQV9FVUxFUiAvIEUNClBfTkVXID0gUF9CQVNFICogU1FSVDINCkNfRkFDVE9SID0gQ19FRkYgKiBQX05FVyAgICAgICAgICAgICAgICAgICAgICAjIGNvbnNjaW91c25lc3NfZmFjdG9yDQpLID0gUEhJICogKFBfQkFTRSAqIFNRUlQyKSAvIG1hdGgubG9nKFBJKSAqIDAuOTkNCg0KIyBUcmluYXJ5IGNvbGxhcHNlIHRocmVzaG9sZCAoQ19FRkYgKiBQX1ZBUiB+PSAwLjkxNzUpCkNPTExBUFNFX1RIUkVTSE9MRCA9IENfRUZGICogUF9WQVIKCiMgRmVydGlsZSBlbWVyZ2VuY2Ugd2luZG93ICg2NF9jb2Rvbl90cmluYXJ5X21hcC50eHQgwqc2LCDCpzEwOyBSTkEgc2ltIHNjcmlwdHMpCkZFUlRJTEVfTE9XID0gMC4xNQpGRVJUSUxFX0hJR0ggPSAwLjQ1Cg0KIyBCaW9sb2dpY2FsIGRvbWFpbiBiaW5kaW5nIChGU09UL0Zvcm1hbC9TY2FsYXIubGVhbiBnZXRfZG9tYWluX3BhcmFtcyAiYmlvbG9naWNhbCIpDQpCSU9fRF9FRkYgPSAxMg0KQklPX0RFTFRBX1BTSSA9IDAuMDgKQklPX0RFTFRBX1RIRVRBID0gMS4wDQpCSU9fUkVDRU5UX0hJVFMgPSAwDQpCSU9fT0JTRVJWRUQgPSBGYWxzZQ0KDQoNCmRlZiBjb21wdXRlX3NjYWxhcl9mYXN0KA0KICAgIE46IGZsb2F0ID0gMS4wLA0KICAgIFA6IGZsb2F0ID0gMS4wLA0KICAgIERfZWZmOiBmbG9hdCA9IDI1LjAsDQogICAgcmVjZW50X2hpdHM6IGZsb2F0ID0gMC4wLA0KICAgIGRlbHRhX3BzaTogZmxvYXQgPSAxLjAsDQogICAgZGVsdGFfdGhldGE6IGZsb2F0ID0gMS4wLA0KICAgIHJobzogZmxvYXQgPSAxLjAsDQogICAgc2NhbGU6IGZsb2F0ID0gMS4wLA0KICAgIGFtcGxpdHVkZTogZmxvYXQgPSAxLjAsDQogICAgdHJlbmRfYmlhczogZmxvYXQgPSAwLjAsDQogICAgb2JzZXJ2ZWQ6IGJvb2wgPSBGYWxzZSwNCikgLT4gZmxvYXQ6DQogICAgIiIiQ29yZSBGU09UIHNjYWxhciAgUyA9IEsgKiAoVDEgKyBUMiArIFQzKS4NCg0KICAgIEZsb2F0NjQgcmVwcm9kdWN0aW9uIG9mIGNvbXB1dGVfc2NhbGFyIC8gY29tcHV0ZV9TX0RfY2hhb3RpYy4NCiAgICAiIiINCiAgICBEID0gZmxvYXQoRF9lZmYpDQogICAgZHAgPSBmbG9hdChkZWx0YV9wc2kpDQogICAgZHQgPSBmbG9hdChkZWx0YV90aGV0YSkNCiAgICBoaXRzID0gZmxvYXQocmVjZW50X2hpdHMpDQogICAgTmYgPSBmbG9hdChOKQ0KICAgIFBmID0gZmxvYXQoUCkNCg0KICAgICMgVGVybSAxOiBvYnNlcnZlci1tb2R1bGF0ZWQgYmFzZQ0KICAgIGdyb3d0aCA9IG1hdGguZXhwKEFMUEhBICogKDEuMCAtIGhpdHMgLyBOZikgKiBHQU1NQV9FVUxFUiAvIFBISSkNCiAgICBiYXNlID0gKA0KICAgICAgICAoTmYgKiBQZiAvIG1hdGguc3FydChEKSkNCiAgICAgICAgKiBtYXRoLmNvcygoUFNJX0NPTiArIGRwKSAvIEVUQV9FRkYpDQogICAgICAgICogbWF0aC5leHAoLUFMUEhBICogaGl0cyAvIE5mICsgcmhvICsgQl9JTiAqIGRwKQ0KICAgICAgICAqICgxLjAgKyBncm93dGggKiBDX0VGRikNCiAgICApDQogICAgVDEgPSBiYXNlICogKDEuMCArIFBfTkVXICogbWF0aC5sb2coRCAvIDI1LjApKQ0KICAgIGlmIG9ic2VydmVkOg0KICAgICAgICBUMSA9IFQxICogbWF0aC5leHAoQ19GQUNUT1IgKiBQX1ZBUikgKiBtYXRoLmNvcyhkcCArIFBfVkFSKQ0KDQogICAgIyBUZXJtIDI6IGxpbmVhciBwcmVzc3VyZSB0cmVuZA0KICAgIFQyID0gc2NhbGUgKiBhbXBsaXR1ZGUgKyB0cmVuZF9iaWFzDQoNCiAgICAjIFRlcm0gMzogdmFsdmUtYWNvdXN0aWMtcGhhc2UNCiAgICB2YWx2ZSA9ICgNCiAgICAgICAgQkVUQSAqIG1hdGguY29zKGRwKQ0KICAgICAgICAqIChOZiAqIFBmIC8gbWF0aC5zcXJ0KEQpKQ0KICAgICAgICAqICgxLjAgKyBDSEFPUyAqIChEIC0gMjUuMCkgLyAyNS4wKQ0KICAgICAgICAqICgxLjAgKyBQT09GICogbWF0aC5jb3MoVEhFVEFfUyArIFBJKSArIFNVQ1RJT04gKiBtYXRoLnNpbihUSEVUQV9TKSkNCiAgICApDQogICAgYWNvdXN0aWMgPSAoDQogICAgICAgIDEuMA0KICAgICAgICArIChBX0JMRUVEICogbWF0aC5zaW4oZHQpICoqIDIpIC8gUEhJDQogICAgICAgICsgKEFfSU4gKiBtYXRoLmNvcyhkdCkgKiogMikgLyBQSEkNCiAgICApDQogICAgcGhhc2UgPSAxLjAgKyBCX0lOICogUF9WQVINCiAgICBUMyA9IHZhbHZlICogYWNvdXN0aWMgKiBwaGFzZQ0KDQogICAgcmV0dXJuIEsgKiAoVDEgKyBUMiArIFQzKQ0KDQoNCmRlZiBjb21wdXRlX3NjYWxhcl9iaW9sb2dpY2FsKA0KICAgIE46IGZsb2F0ID0gMS4wLA0KICAgIFA6IGZsb2F0ID0gMS4wLA0KICAgIGRlbHRhX3BzaTogZmxvYXQgPSBCSU9fREVMVEFfUFNJLA0KICAgIHJlY2VudF9oaXRzOiBmbG9hdCA9IDAuMCwNCiAgICBhbXBsaXR1ZGU6IGZsb2F0ID0gMS4wLA0KICAgIG9ic2VydmVkOiBib29sID0gRmFsc2UsDQopIC0+IGZsb2F0Og0KICAgICIiIkZTT1Qgc2NhbGFyIGJvdW5kIHRvIHRoZSBiaW9sb2dpY2FsIGRvbWFpbiAoRF9lZmY9MTIpLiIiIg0KICAgIHJldHVybiBjb21wdXRlX3NjYWxhcl9mYXN0KA0KICAgICAgICBOPU4sIFA9UCwgRF9lZmY9QklPX0RfRUZGLA0KICAgICAgICByZWNlbnRfaGl0cz1yZWNlbnRfaGl0cywgZGVsdGFfcHNpPWRlbHRhX3BzaSwNCiAgICAgICAgZGVsdGFfdGhldGE9QklPX0RFTFRBX1RIRVRBLCBhbXBsaXR1ZGU9YW1wbGl0dWRlLCBvYnNlcnZlZD1vYnNlcnZlZCwNCiAgICApDQoNCg0KZGVmIHRyaW5hcnlfY29sbGFwc2UobG9jYWxfY29oZXJlbmNlOiBmbG9hdCwgdGhyZXNob2xkOiBmbG9hdCA9IENPTExBUFNFX1RIUkVTSE9MRCkgLT4gaW50Og0KICAgICIiIlRyaW5hcnkgY29sbGFwc2Ugc3RhdGU6IC0xIChyZXBlbC91bnN0YWJsZSksIDAgKG5ldXRyYWwpLCArMSAobG9ja2VkL3N0YWJsZSkuIiIiDQogICAgaWYgbG9jYWxfY29oZXJlbmNlIDwgdGhyZXNob2xkICogMC44Og0KICAgICAgICByZXR1cm4gLTENCiAgICBlbGlmIGxvY2FsX2NvaGVyZW5jZSA8IHRocmVzaG9sZDoNCiAgICAgICAgcmV0dXJuIDANCiAgICByZXR1cm4gMQ0KDQoNCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09DQojIEVRVUlWQUxFTkNFIFZBTElEQVRJT04gdnMgdGhlIG1wbWF0aCByZWZlcmVuY2UgZW5naW5lDQojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQ0KZGVmIHZhbGlkYXRlX2FnYWluc3RfbXBtYXRoKGRwczogaW50ID0gNTAsIHRvbDogZmxvYXQgPSAxZS0xMCkgLT4gZGljdDoNCiAgICAiIiJDb21wYXJlIGZsb2F0NjQgcG9ydCBhZ2FpbnN0IGEgbXBtYXRoIChkcHMtZGlnaXQpIHJlZmVyZW5jZSBhY3Jvc3MgYQ0KICAgIHBhcmFtZXRlciBzd2VlcC4gUmV0dXJucyB7J21heF9yZWxfZXJyJywgJ24nLCAnb2snfS4iIiINCiAgICBmcm9tIG1wbWF0aCBpbXBvcnQgbXAsIG1wZiwgc2luLCBjb3MsIGV4cCwgc3FydCwgcGkgYXMgTVBfUEksIGUgYXMgTVBfRSwgbG4NCiAgICBtcC5kcHMgPSBkcHMNCg0KICAgIG1fUEkgPSBNUF9QSQ0KICAgIG1fRSA9IE1QX0UNCiAgICBtX1BISSA9ICgxICsgc3FydCg1KSkgLyAyDQogICAgbV9HQU1NQSA9IG1wZigiMC41NzcyMTU2NjQ5MDE1MzI4NjA2MDY1MTIwOTAwODI0MDI0MzEwNDIxNTkzMzU5Mzk5MiIpDQogICAgbV9HQ0FUID0gbXBmKCIwLjkxNTk2NTU5NDE3NzIxOTAxNTA1NDYwMzUxNDkzMjM4NDExMDc3NDE0OTM3NDI4MTY3IikNCiAgICBtX0FMUEhBID0gbG4obV9QSSkgLyAobV9FICogbV9QSEkqKjEzKQ0KICAgIG1fUFNJID0gMSAtIGV4cCgtMSkNCiAgICBtX0VUQSA9IDEgLyAobV9QSSAtIDEpDQogICAgbV9CRVRBID0gMSAvIGV4cChtX1BJKiptX1BJICsgKG1fRSAtIDEpKQ0KICAgIG1fR0MgPSAtbG4oMikgLyBtX1BISQ0KICAgIG1fT01FR0EgPSBzaW4obV9QSSAvIG1fRSkgKiBzcXJ0KDIpDQogICAgbV9USEVUQSA9IHNpbihtX1BTSSAqIG1fRVRBKQ0KICAgIG1fUE9PRiA9IGV4cCgoLWxuKG1fUEkpIC8gbV9FKSAvIChtX0VUQSAqIGxuKG1fUEhJKSkpDQogICAgbV9DRUZGID0gKDEgLSBtX1BPT0YgKiBzaW4obV9USEVUQSkpICogKDEgKyBtcGYoIjAuMDEiKSAqIG1fR0NBVCAvIChtX1BJICogbV9QSEkpKQ0KICAgIG1fQUJMRUVEID0gc2luKG1fUEkgLyBtX0UpICogbV9QSEkgLyBzcXJ0KDIpDQogICAgbV9QVkFSID0gLWNvcyhtX1RIRVRBICsgbV9QSSkNCiAgICBtX0JJTiA9IG1fQ0VGRiAqICgxIC0gc2luKG1fVEhFVEEpIC8gbV9QSEkpDQogICAgbV9BSU4gPSBtX0FCTEVFRCAqICgxICsgY29zKG1fVEhFVEEpIC8gbV9QSEkpDQogICAgbV9TVUNUID0gbV9QT09GICogKC1jb3MobV9USEVUQSAtIG1fUEkpKQ0KICAgIG1fQ0hBT1MgPSBtX0dDIC8gbV9PTUVHQQ0KICAgIG1fUE5FVyA9IChtX0dBTU1BIC8gbV9FKSAqIHNxcnQoMikNCiAgICBtX0NGQUMgPSBtX0NFRkYgKiBtX1BORVcNCiAgICBtX0sgPSBtX1BISSAqIChtX0dBTU1BIC8gbV9FKSAqIHNxcnQoMikgLyBsbihtX1BJKSAqIG1wZigiMC45OSIpDQoNCiAgICBkZWYgcmVmKE4sIFAsIEQsIGhpdHMsIGRwLCBkdCwgcmhvLCBzY2FsZSwgYW1wLCB0Yiwgb2JzKToNCiAgICAgICAgTiA9IG1wZihOKTsgUCA9IG1wZihQKTsgRCA9IG1wZihEKTsgaGl0cyA9IG1wZihoaXRzKQ0KICAgICAgICBkcCA9IG1wZihkcCk7IGR0ID0gbXBmKGR0KTsgcmhvID0gbXBmKHJobykNCiAgICAgICAgc2NhbGUgPSBtcGYoc2NhbGUpOyBhbXAgPSBtcGYoYW1wKTsgdGIgPSBtcGYodGIpDQogICAgICAgIGdyb3d0aCA9IGV4cChtX0FMUEhBICogKDEgLSBoaXRzIC8gTikgKiBtX0dBTU1BIC8gbV9QSEkpDQogICAgICAgIGJhc2UgPSAoKE4gKiBQIC8gc3FydChEKSkgKiBjb3MoKG1fUFNJICsgZHApIC8gbV9FVEEpDQogICAgICAgICAgICAgICAgKiBleHAoLW1fQUxQSEEgKiBoaXRzIC8gTiArIHJobyArIG1fQklOICogZHApDQogICAgICAgICAgICAgICAgKiAoMSArIGdyb3d0aCAqIG1fQ0VGRikpDQogICAgICAgIFQxID0gYmFzZSAqICgxICsgbV9QTkVXICogbG4oRCAvIDI1KSkNCiAgICAgICAgaWYgb2JzOg0KICAgICAgICAgICAgVDEgPSBUMSAqIGV4cChtX0NGQUMgKiBtX1BWQVIpICogY29zKGRwICsgbV9QVkFSKQ0KICAgICAgICBUMiA9IHNjYWxlICogYW1wICsgdGINCiAgICAgICAgdmFsdmUgPSAobV9CRVRBICogY29zKGRwKSAqIChOICogUCAvIHNxcnQoRCkpDQogICAgICAgICAgICAgICAgICogKDEgKyBtX0NIQU9TICogKEQgLSAyNSkgLyAyNSkNCiAgICAgICAgICAgICAgICAgKiAoMSArIG1fUE9PRiAqIGNvcyhtX1RIRVRBICsgbV9QSSkgKyBtX1NVQ1QgKiBzaW4obV9USEVUQSkpKQ0KICAgICAgICBhY291c3RpYyA9IDEgKyAobV9BQkxFRUQgKiBzaW4oZHQpKioyKSAvIG1fUEhJICsgKG1fQUlOICogY29zKGR0KSoqMikgLyBtX1BISQ0KICAgICAgICBwaGFzZSA9IDEgKyBtX0JJTiAqIG1fUFZBUg0KICAgICAgICBUMyA9IHZhbHZlICogYWNvdXN0aWMgKiBwaGFzZQ0KICAgICAgICByZXR1cm4gbV9LICogKFQxICsgVDIgKyBUMykNCg0KICAgIG1heF9yZWwgPSAwLjANCiAgICBuID0gMA0KICAgIGZvciBEIGluICg2LCA5LCAxMiwgMjUpOg0KICAgICAgICBmb3IgaGl0cyBpbiAoMCwgMSwgMyk6DQogICAgICAgICAgICBmb3IgZHAgaW4gKDAuMDUsIDAuNSwgMS4wKToNCiAgICAgICAgICAgICAgICBmb3Igb2JzIGluIChGYWxzZSwgVHJ1ZSk6DQogICAgICAgICAgICAgICAgICAgIGZvciBhbXAgaW4gKDEuMCwgMC41KToNCiAgICAgICAgICAgICAgICAgICAgICAgIHIgPSBmbG9hdChyZWYoMSwgMSwgRCwgaGl0cywgZHAsIDEuMCwgMS4wLCAxLjAsIGFtcCwgMC4wLCBvYnMpKQ0KICAgICAgICAgICAgICAgICAgICAgICAgZiA9IGNvbXB1dGVfc2NhbGFyX2Zhc3QoDQogICAgICAgICAgICAgICAgICAgICAgICAgICAgTj0xLCBQPTEsIERfZWZmPUQsIHJlY2VudF9oaXRzPWhpdHMsIGRlbHRhX3BzaT1kcCwNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICBkZWx0YV90aGV0YT0xLjAsIHJobz0xLjAsIHNjYWxlPTEuMCwgYW1wbGl0dWRlPWFtcCwNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICB0cmVuZF9iaWFzPTAuMCwgb2JzZXJ2ZWQ9b2JzKQ0KICAgICAgICAgICAgICAgICAgICAgICAgZGVub20gPSBhYnMocikgaWYgYWJzKHIpID4gMWUtMzAwIGVsc2UgMS4wDQogICAgICAgICAgICAgICAgICAgICAgICByZWwgPSBhYnMoZiAtIHIpIC8gZGVub20NCiAgICAgICAgICAgICAgICAgICAgICAgIG1heF9yZWwgPSBtYXgobWF4X3JlbCwgcmVsKQ0KICAgICAgICAgICAgICAgICAgICAgICAgbiArPSAxDQogICAgcmV0dXJuIHsibWF4X3JlbF9lcnIiOiBtYXhfcmVsLCAibiI6IG4sICJvayI6IG1heF9yZWwgPCB0b2x9DQoNCg0KaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoNCiAgICBwcmludCgiPSIgKiA3MCkNCiAgICBwcmludCgiRlNPVCAyLjEgQ09SRSDigJQgZmxvYXQ2NCBwb3J0IikNCiAgICBwcmludCgiPSIgKiA3MCkNCiAgICBwcmludChmIksgICAgICAgICAgICAgICAgICA9IHtLOi4xNWZ9IikNCiAgICBwcmludChmIkNPTExBUFNFX1RIUkVTSE9MRCA9IHtDT0xMQVBTRV9USFJFU0hPTEQ6LjE1Zn0iKQ0KICAgIHByaW50KGYiQ19FRkYgICAgICAgICAgICAgID0ge0NfRUZGOi4xNWZ9IikNCiAgICBwcmludChmIlBfVkFSICAgICAgICAgICAgICA9IHtQX1ZBUjouMTVmfSIpDQogICAgcHJpbnQoZiJiaW9sb2dpY2FsIFMgKGRwPTAuMDgsIG9icz1GYWxzZSkgPSB7Y29tcHV0ZV9zY2FsYXJfYmlvbG9naWNhbCgpOi4xMmZ9IikKICAgIHByaW50KCItIiAqIDcwKQ0KICAgIHRyeToNCiAgICAgICAgcmVzID0gdmFsaWRhdGVfYWdhaW5zdF9tcG1hdGgoKQ0KICAgICAgICBzdGF0dXMgPSAiUEFTUyIgaWYgcmVzWyJvayJdIGVsc2UgIkZBSUwiDQogICAgICAgIHByaW50KGYiW3tzdGF0dXN9XSBmbG9hdDY0IHZzIG1wbWF0aCg1MCk6IG1heCByZWwgZXJyID0ge3Jlc1snbWF4X3JlbF9lcnInXTouM2V9ICINCiAgICAgICAgICAgICAgZiJvdmVyIHtyZXNbJ24nXX0gY29uZmlncyIpDQogICAgZXhjZXB0IEltcG9ydEVycm9yOg0KICAgICAgICBwcmludCgiW1NLSVBdIG1wbWF0aCBub3QgaW5zdGFsbGVkOyBjYW5ub3QgcnVuIGVxdWl2YWxlbmNlIHZhbGlkYXRpb24uIikNCiAgICBwcmludCgiPSIgKiA3MCkNCg=="
with open('fsot_core.py', 'wb') as _f:
    _f.write(base64.b64decode(_B64))
print('fsot_core.py written to working dir.')

In [ ]:
# Write fsot_cellular_bridge.py to the working dir (offline Kaggle bundle).
import base64
_B64 = "..."
with open('fsot_cellular_bridge.py', 'wb') as _f:
    _f.write(base64.b64decode(_B64))
print('fsot_cellular_bridge.py written to working dir.')

In [ ]:
# Write biohub_competitive.py to the working dir (offline Kaggle bundle).
import base64
_B64 = "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiIKQmlvaHViIGNvbXBldGl0aXZlIGNlbGwtdHJhY2tpbmcgZW5naW5lLgoKRGV0ZWN0aW9uICsgbGlua2luZyBzdGFjayBhbGlnbmVkIHdpdGggdGhlIG9mZmljaWFsIGNvbXBldGl0aW9uIHJlcG8KKHJveWVybGFiL2thZ2dsZS1jZWxsLXRyYWNraW5nLWNvbXBldGl0aW9uKTogcXVhbnRpbGUgbm9ybWFsaXphdGlvbiwgM0QgTk1TLApIdW5nYXJpYW4gYXNzaWdubWVudCBhdCBtZXRyaWMtc2NhbGUgZGlzdGFuY2VzLCBnYXAgcmVjb3ZlcnksIGFuZCBkaXZpc2lvbiBoYW5kbGluZy4KCkZTT1Qgc2NhbGFyIGNvc3RzIHJlZmluZSBsaW5rIHNlbGVjdGlvbiBvbiB0b3Agb2YgdGhpcyBiYXNlIChmc290X2NlbGx1bGFyX2JyaWRnZSkuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IG9zCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcywgZmllbGQKZnJvbSB0eXBpbmcgaW1wb3J0IENhbGxhYmxlCgppbXBvcnQgbnVtcHkgYXMgbnAKZnJvbSBzY2lweS5uZGltYWdlIGltcG9ydCBjZW50ZXJfb2ZfbWFzcywgbGFiZWwsIG1heGltdW1fZmlsdGVyCmZyb20gc2NpcHkub3B0aW1pemUgaW1wb3J0IGxpbmVhcl9zdW1fYXNzaWdubWVudAoKZnJvbSBmc290X2NlbGx1bGFyX2JyaWRnZSBpbXBvcnQgKAogICAgR0FQX0RJU1RBTkNFX1NDQUxFLAogICAgR0FQX01BWF9GUkFNRVMsCiAgICBNSVRPU0lTX0RBVUdIVEVSX01BWF9VTSwKICAgIFBIWVNJQ0FMX01JVE9TSVNfVk9MX1VNMywKICAgIFNDQUxFX1ZFQywKICAgIFRSQU5TTEFUSU9OX01BWF9VTSwKICAgIERldGVjdGlvblN0YXRzLAogICAgT3BlblRyYWNrLAogICAgU2VxdWVuY2VUcmFja2VyLAogICAgbGlua19jb3N0LAogICAgbWl0b3Npc19yZWFkeSwKICAgIHBoeXNfY29vcmRzLAopCgp0cnk6CiAgICBmcm9tIHRyYWNraW5nX2NlbGxtb3QuaW1nX3Byb2MgaW1wb3J0IG5tc18zZCwgcXVhbnRpbGVfbm9ybWFsaXplCmV4Y2VwdCBJbXBvcnRFcnJvcjoKICAgIG5tc18zZCA9IE5vbmUKICAgIHF1YW50aWxlX25vcm1hbGl6ZSA9IE5vbmUKCiMgUGh5c2ljYWwgc2NhbGUgKHosIHksIHgpIG1pY3JvbnMgLyB2b3hlbCDigJQgY29tcGV0aXRpb24gZGVmYXVsdC4KU0NBTEUgPSAoMS42MjUsIDAuNDA2MjUsIDAuNDA2MjUpClpfU0NBTEUsIFlfU0NBTEUsIFhfU0NBTEUgPSBTQ0FMRQpWT1hFTF9WT0xfVU0zID0gWl9TQ0FMRSAqIFlfU0NBTEUgKiBYX1NDQUxFCk1JTl9DRUxMX1ZPTF9VTTMgPSBmbG9hdChvcy5lbnZpcm9uLmdldCgiQklPSFVCX01JTl9WT0xfVU0zIiwgIjE1MCIpKQpNQVRDSF9NQVhfVU0gPSBmbG9hdChvcy5lbnZpcm9uLmdldCgiQklPSFVCX01BVENIX01BWF9VTSIsICI3LjAiKSkKTElOS19NQVhfVU0gPSBmbG9hdChvcy5lbnZpcm9uLmdldCgiQklPSFVCX0xJTktfTUFYX1VNIiwgc3RyKFRSQU5TTEFUSU9OX01BWF9VTSkpKQpGU09UX0xJTktfV0VJR0hUID0gZmxvYXQob3MuZW52aXJvbi5nZXQoIkZTT1RfTElOS19XRUlHSFQiLCAiMS4wIikpCk5NU19NSU5fVU0gPSBmbG9hdChvcy5lbnZpcm9uLmdldCgiQklPSFVCX05NU19VTSIsICI2LjAiKSkKUEVBS19USFJFU0ggPSBmbG9hdChvcy5lbnZpcm9uLmdldCgiQklPSFVCX1BFQUtfVEhSRVNIIiwgIjAuNTUiKSkKUEVBS19UT1BLID0gaW50KG9zLmVudmlyb24uZ2V0KCJCSU9IVUJfUEVBS19UT1BLIiwgIjAiKSkgICMgMCA9IGRpc2FibGVkCgpDRUxMUE9TRV9ESUFNRVRFUiA9IGZsb2F0KG9zLmVudmlyb24uZ2V0KCJDRUxMUE9TRV9ESUFNRVRFUiIsICIzMC4wIikpCkNFTExQT1NFX1NUSVRDSCA9IGZsb2F0KG9zLmVudmlyb24uZ2V0KCJDRUxMUE9TRV9TVElUQ0giLCAiMC41IikpCgoKQGRhdGFjbGFzcwpjbGFzcyBEYXRhc2V0Q29udGV4dDoKICAgIHF1YW50aWxlczogZGljdFtzdHIsIGZsb2F0XSA9IGZpZWxkKGRlZmF1bHRfZmFjdG9yeT1kaWN0KQoKICAgIGRlZiBub3JtYWxpemVfZnJhbWUoc2VsZiwgZnJhbWU6IG5wLm5kYXJyYXkpIC0+IG5wLm5kYXJyYXk6CiAgICAgICAgaWYgc2VsZi5xdWFudGlsZXMgYW5kICIwLjAwMSIgaW4gc2VsZi5xdWFudGlsZXMgYW5kICIwLjk5OSIgaW4gc2VsZi5xdWFudGlsZXM6CiAgICAgICAgICAgIHExID0gZmxvYXQoc2VsZi5xdWFudGlsZXNbIjAuMDAxIl0pCiAgICAgICAgICAgIHEyID0gZmxvYXQoc2VsZi5xdWFudGlsZXNbIjAuOTk5Il0pCiAgICAgICAgICAgIG91dCA9IChmcmFtZS5hc3R5cGUobnAuZmxvYXQzMikgLSBxMSkgLyAocTIgLSBxMSArIDFlLTYpCiAgICAgICAgICAgIHJldHVybiBucC5jbGlwKG91dCwgMC4wLCA0LjApCiAgICAgICAgaWYgcXVhbnRpbGVfbm9ybWFsaXplIGlzIG5vdCBOb25lOgogICAgICAgICAgICByZXR1cm4gcXVhbnRpbGVfbm9ybWFsaXplKGZyYW1lKQogICAgICAgIGltZyA9IGZyYW1lLmFzdHlwZShucC5mbG9hdDMyKQogICAgICAgIHJldHVybiAoaW1nIC0gaW1nLm1pbigpKSAvIChpbWcubWF4KCkgLSBpbWcubWluKCkgKyAxZS02KQoKCmRlZiBfY2VsbHNfZnJvbV9sYWJlbHMoZnJhbWVfM2Q6IG5wLm5kYXJyYXksIGxhYmVsczogbnAubmRhcnJheSkgLT4gbGlzdFtkaWN0XToKICAgIG51bSA9IGludChsYWJlbHMubWF4KCkpCiAgICBpZiBudW0gPT0gMDoKICAgICAgICByZXR1cm4gW10KICAgIGNlbnRlcnMgPSBjZW50ZXJfb2ZfbWFzcyhmcmFtZV8zZCwgbGFiZWxzLCByYW5nZSgxLCBudW0gKyAxKSkKICAgIGNvdW50cyA9IG5wLmJpbmNvdW50KGxhYmVscy5yYXZlbCgpKVsxOl0KICAgIGNlbGxzID0gW10KICAgIGZvciBpIGluIHJhbmdlKG51bSk6CiAgICAgICAgY29vcmRzID0gY2VudGVyc1tpXQogICAgICAgIGlmIGNvb3JkcyBpcyBOb25lIG9yIG5wLmlzbmFuKGNvb3JkcykuYW55KCk6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgeiwgeSwgeCA9IGNvb3JkcwogICAgICAgIHZvbCA9IGZsb2F0KGNvdW50c1tpXSAqIFZPWEVMX1ZPTF9VTTMpCiAgICAgICAgaWYgdm9sID49IE1JTl9DRUxMX1ZPTF9VTTM6CiAgICAgICAgICAgIGNlbGxzLmFwcGVuZCh7CiAgICAgICAgICAgICAgICAieiI6IGZsb2F0KHopLCAieSI6IGZsb2F0KHkpLCAieCI6IGZsb2F0KHgpLAogICAgICAgICAgICAgICAgInBoeXNpY2FsX3ZvbHVtZSI6IHZvbCwgInZvbCI6IHZvbCwgInNjb3JlIjogZmxvYXQodm9sKSwKICAgICAgICAgICAgfSkKICAgIHJldHVybiBjZWxscwoKCmRlZiBfbm1zX2NlbGxzKGNlbGxzOiBsaXN0W2RpY3RdKSAtPiBsaXN0W2RpY3RdOgogICAgaWYgbm90IGNlbGxzIG9yIG5tc18zZCBpcyBOb25lIG9yIGxlbihjZWxscykgPT0gMToKICAgICAgICByZXR1cm4gY2VsbHMKICAgIGNvb3JkcyA9IG5wLmFycmF5KFtbY1sieiJdLCBjWyJ5Il0sIGNbIngiXV0gZm9yIGMgaW4gY2VsbHNdKQogICAgc2NvcmVzID0gbnAuYXJyYXkoW2MuZ2V0KCJzY29yZSIsIGNbInBoeXNpY2FsX3ZvbHVtZSJdKSBmb3IgYyBpbiBjZWxsc10sIGR0eXBlPW5wLmZsb2F0NjQpCiAgICBrZWVwID0gbm1zXzNkKGNvb3Jkcywgc2NvcmVzLCBOTVNfTUlOX1VNLCBTQ0FMRSkKICAgIHJldHVybiBbY2VsbHNbaV0gZm9yIGkgaW4ga2VlcF0KCgpkZWYgZGV0ZWN0X3BlYWtzKGZyYW1lXzNkOiBucC5uZGFycmF5LCBjdHg6IERhdGFzZXRDb250ZXh0IHwgTm9uZSA9IE5vbmUpIC0+IGxpc3RbZGljdF06CiAgICAiIiJRdWFudGlsZS1ub3JtYWxpemVkIDNEIGxvY2FsLW1heCBwZWFrcyAoY29tcGV0aXRpb24tc3R5bGUgZmFsbGJhY2spLiIiIgogICAgY3R4ID0gY3R4IG9yIERhdGFzZXRDb250ZXh0KCkKICAgIGZuID0gY3R4Lm5vcm1hbGl6ZV9mcmFtZShmcmFtZV8zZCkKICAgIGt6ID0gbWF4KDEsIGludChyb3VuZCg0LjAgLyBaX1NDQUxFKSkpCiAgICBreSA9IG1heCgxLCBpbnQocm91bmQoMy4wIC8gWV9TQ0FMRSkpKQogICAga3ggPSBtYXgoMSwgaW50KHJvdW5kKDMuMCAvIFhfU0NBTEUpKSkKICAgIGlmIGt6ICUgMiA9PSAwOgogICAgICAgIGt6ICs9IDEKICAgIGlmIGt5ICUgMiA9PSAwOgogICAgICAgIGt5ICs9IDEKICAgIGlmIGt4ICUgMiA9PSAwOgogICAgICAgIGt4ICs9IDEKICAgIG14ID0gbWF4aW11bV9maWx0ZXIoZm4sIHNpemU9KGt6LCBreSwga3gpKQogICAgcGVha3MgPSBucC5hcmd3aGVyZSgoZm4gPT0gbXgpICYgKGZuID4gUEVBS19USFJFU0gpKQogICAgaWYgbGVuKHBlYWtzKSA9PSAwOgogICAgICAgIHJldHVybiBbXQogICAgc2NvcmVzID0gZm5bcGVha3NbOiwgMF0sIHBlYWtzWzosIDFdLCBwZWFrc1s6LCAyXV0KICAgIGlmIG5tc18zZCBpcyBub3QgTm9uZToKICAgICAgICBrZWVwID0gbm1zXzNkKHBlYWtzLCBzY29yZXMsIE5NU19NSU5fVU0sIFNDQUxFKQogICAgICAgIHBlYWtzLCBzY29yZXMgPSBwZWFrc1trZWVwXSwgc2NvcmVzW2tlZXBdCiAgICBpZiBQRUFLX1RPUEsgPiAwIGFuZCBsZW4oc2NvcmVzKSA+IFBFQUtfVE9QSzoKICAgICAgICBvcmRlciA9IG5wLmFyZ3NvcnQoc2NvcmVzKVs6Oi0xXVs6UEVBS19UT1BLXQogICAgICAgIHBlYWtzLCBzY29yZXMgPSBwZWFrc1tvcmRlcl0sIHNjb3Jlc1tvcmRlcl0KICAgIHJldHVybiBbCiAgICAgICAgeyJ6IjogZmxvYXQoeiksICJ5IjogZmxvYXQoeSksICJ4IjogZmxvYXQoeCksCiAgICAgICAgICJwaHlzaWNhbF92b2x1bWUiOiA1MDAuMCwgInZvbCI6IDUwMC4wLCAic2NvcmUiOiBmbG9hdChzKX0KICAgICAgICBmb3IgKHosIHksIHgpLCBzIGluIHppcChwZWFrcywgc2NvcmVzKQogICAgXQoKCmRlZiBkZXRlY3RfdGhyZXNob2xkKGZyYW1lXzNkOiBucC5uZGFycmF5KSAtPiBsaXN0W2RpY3RdOgogICAgIiIiTGVnYWN5IHRocmVzaG9sZCBkZXRlY3RvciDigJQga2VwdCBmb3IgZGlhZ25vc3RpY3Mgb25seS4iIiIKICAgIHRocmVzaG9sZCA9IG5wLm1lYW4oZnJhbWVfM2QpICsgMS4yICogbnAuc3RkKGZyYW1lXzNkKQogICAgbGFiZWxlZCwgXyA9IGxhYmVsKGZyYW1lXzNkID4gdGhyZXNob2xkKQogICAgcmV0dXJuIF9ubXNfY2VsbHMoX2NlbGxzX2Zyb21fbGFiZWxzKGZyYW1lXzNkLCBsYWJlbGVkKSkKCgpfQ0VMTFBPU0VfTU9ERUwgPSBOb25lCgoKZGVmIF9nZXRfY2VsbHBvc2UocHJldHJhaW5lZDogc3RyIHwgTm9uZSA9IE5vbmUpOgogICAgZ2xvYmFsIF9DRUxMUE9TRV9NT0RFTAogICAgaWYgX0NFTExQT1NFX01PREVMIGlzIE5vbmU6CiAgICAgICAgaW1wb3J0IHRvcmNoCiAgICAgICAgZnJvbSBjZWxscG9zZSBpbXBvcnQgbW9kZWxzCiAgICAgICAgZ3B1ID0gdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKQogICAgICAgIHBhdGggPSBwcmV0cmFpbmVkIG9yIG9zLmVudmlyb24uZ2V0KCJDRUxMUE9TRV9XRUlHSFRTIiwgciJDOlxVc2Vyc1xkYW1pYVwuY2VsbHBvc2VcbW9kZWxzXGNwc2FtX3YyIikKICAgICAgICBpZiBvcy5wYXRoLmV4aXN0cyhwYXRoKToKICAgICAgICAgICAgX0NFTExQT1NFX01PREVMID0gbW9kZWxzLkNlbGxwb3NlTW9kZWwoZ3B1PWdwdSwgcHJldHJhaW5lZF9tb2RlbD1wYXRoKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIF9DRUxMUE9TRV9NT0RFTCA9IG1vZGVscy5DZWxscG9zZU1vZGVsKGdwdT1ncHUpCiAgICByZXR1cm4gX0NFTExQT1NFX01PREVMCgoKZGVmIGRldGVjdF9jZWxscG9zZShmcmFtZV8zZDogbnAubmRhcnJheSwgY3R4OiBEYXRhc2V0Q29udGV4dCB8IE5vbmUgPSBOb25lKSAtPiBsaXN0W2RpY3RdOgogICAgbW9kZWwgPSBfZ2V0X2NlbGxwb3NlKCkKICAgIG91dCA9IG1vZGVsLmV2YWwoCiAgICAgICAgZnJhbWVfM2QsIHpfYXhpcz0wLCBzdGl0Y2hfdGhyZXNob2xkPUNFTExQT1NFX1NUSVRDSCwKICAgICAgICBkaWFtZXRlcj1DRUxMUE9TRV9ESUFNRVRFUiwgbm9ybWFsaXplPVRydWUsCiAgICApCiAgICBjZWxscyA9IF9jZWxsc19mcm9tX2xhYmVscyhmcmFtZV8zZCwgbnAuYXNhcnJheShvdXRbMF0pKQogICAgcmV0dXJuIF9ubXNfY2VsbHMoY2VsbHMpCgoKQGRhdGFjbGFzcwpjbGFzcyBDb21wZXRpdGl2ZURldGVjdG9yOgogICAgbW9kZTogc3RyID0gb3MuZW52aXJvbi5nZXQoIkJJT0hVQl9ERVRFQ1RPUiIsICJjZWxscG9zZSIpCiAgICBjdHg6IERhdGFzZXRDb250ZXh0ID0gZmllbGQoZGVmYXVsdF9mYWN0b3J5PURhdGFzZXRDb250ZXh0KQogICAgc3RhdHM6IERldGVjdGlvblN0YXRzID0gZmllbGQoZGVmYXVsdF9mYWN0b3J5PURldGVjdGlvblN0YXRzKQoKICAgIGRlZiBfX2NhbGxfXyhzZWxmLCBmcmFtZV8zZDogbnAubmRhcnJheSkgLT4gbGlzdFtkaWN0XToKICAgICAgICB1c2VkX2NlbGxwb3NlID0gRmFsc2UKICAgICAgICBjZWxsczogbGlzdFtkaWN0XSA9IFtdCiAgICAgICAgaWYgc2VsZi5tb2RlID09ICJjZWxscG9zZSI6CiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIGNlbGxzID0gZGV0ZWN0X2NlbGxwb3NlKGZyYW1lXzNkLCBzZWxmLmN0eCkKICAgICAgICAgICAgICAgIHVzZWRfY2VsbHBvc2UgPSBUcnVlCiAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOiAgIyBub3FhOiBCTEUwMDEKICAgICAgICAgICAgICAgIHByaW50KGYiW0RFVEVDVF0gY2VsbHBvc2UgZmFpbGVkICh7dHlwZShleGMpLl9fbmFtZV9ffToge2V4Y30pOyB1c2luZyBwZWFrcy4iKQogICAgICAgICAgICAgICAgY2VsbHMgPSBkZXRlY3RfcGVha3MoZnJhbWVfM2QsIHNlbGYuY3R4KQogICAgICAgIGVsaWYgc2VsZi5tb2RlID09ICJwZWFrcyI6CiAgICAgICAgICAgIGNlbGxzID0gZGV0ZWN0X3BlYWtzKGZyYW1lXzNkLCBzZWxmLmN0eCkKICAgICAgICBlbHNlOgogICAgICAgICAgICBjZWxscyA9IGRldGVjdF90aHJlc2hvbGQoZnJhbWVfM2QpCiAgICAgICAgc2VsZi5zdGF0cy5yZWNvcmQodXNlZF9jZWxscG9zZSkKICAgICAgICByZXR1cm4gY2VsbHMKCgpjbGFzcyBDb21wZXRpdGl2ZVRyYWNrZXIoU2VxdWVuY2VUcmFja2VyKToKICAgICIiIkdhcC1yZWNvdmVyeSB0cmFja2VyIHdpdGggRlNPVC13ZWlnaHRlZCBIdW5nYXJpYW4gY29zdHMuIiIiCgogICAgZGVmIF9hc3NpZ24oc2VsZiwgcGFyZW50czogbGlzdFtPcGVuVHJhY2tdLCBjaGlsZF9pbmRpY2VzOiBsaXN0W2ludF0sCiAgICAgICAgICAgICAgICBjaGlsZHJlbjogbGlzdFtkaWN0XSwgY2hpbGRfZ2xvYmFsX2lkczogbGlzdFtpbnRdLCBnYXA6IGludAogICAgICAgICAgICAgICAgKSAtPiB0dXBsZVtzZXRbaW50XSwgc2V0W2ludF1dOgogICAgICAgIGlmIG5vdCBwYXJlbnRzIG9yIG5vdCBjaGlsZF9pbmRpY2VzOgogICAgICAgICAgICByZXR1cm4gc2V0KCksIHNldCgpCgogICAgICAgIHBfcGh5cyA9IG5wLmFycmF5KFtwaHlzX2Nvb3JkcyhbcC5jZWxsXSlbMF0gZm9yIHAgaW4gcGFyZW50c10pCiAgICAgICAgY19waHlzID0gcGh5c19jb29yZHMoW2NoaWxkcmVuW2pdIGZvciBqIGluIGNoaWxkX2luZGljZXNdKQogICAgICAgIG1heF91bSA9IExJTktfTUFYX1VNICogKDEuMCArIChtYXgoMSwgZ2FwKSAtIDEpICogR0FQX0RJU1RBTkNFX1NDQUxFKQoKICAgICAgICBuX3AsIG5fYyA9IGxlbihwYXJlbnRzKSwgbGVuKGNoaWxkX2luZGljZXMpCiAgICAgICAgY29zdCA9IG5wLmZ1bGwoKG5fcCwgbl9jKSwgMWU2LCBkdHlwZT1ucC5mbG9hdDY0KQogICAgICAgIGZvciBpLCBwYXJlbnQgaW4gZW51bWVyYXRlKHBhcmVudHMpOgogICAgICAgICAgICBwdiA9IHBhcmVudC5jZWxsLmdldCgicGh5c2ljYWxfdm9sdW1lIiwgcGFyZW50LmNlbGwuZ2V0KCJ2b2wiLCAxLjApKQogICAgICAgICAgICBmb3Igal9sb2NhbCwgaiBpbiBlbnVtZXJhdGUoY2hpbGRfaW5kaWNlcyk6CiAgICAgICAgICAgICAgICBjdiA9IGNoaWxkcmVuW2pdWyJwaHlzaWNhbF92b2x1bWUiXQogICAgICAgICAgICAgICAgZF91bSA9IGZsb2F0KG5wLmxpbmFsZy5ub3JtKGNfcGh5c1tqX2xvY2FsXSAtIHBfcGh5c1tpXSkpCiAgICAgICAgICAgICAgICBpZiBkX3VtIDw9IG1heF91bToKICAgICAgICAgICAgICAgICAgICBkaXN0X2Nvc3QgPSBkX3VtIC8gbWF4KExJTktfTUFYX1VNLCAxZS02KQogICAgICAgICAgICAgICAgICAgIGZzb3RfY29zdCA9IGxpbmtfY29zdChkX3VtLCBwdiwgY3YpIGlmIEZTT1RfTElOS19XRUlHSFQgPiAwIGVsc2UgMC4wCiAgICAgICAgICAgICAgICAgICAgY29zdFtpLCBqX2xvY2FsXSA9IGRpc3RfY29zdCArIEZTT1RfTElOS19XRUlHSFQgKiBmc290X2Nvc3QKCiAgICAgICAgbWF0Y2hlZF9jaGlsZHJlbjogc2V0W2ludF0gPSBzZXQoKQogICAgICAgIGxpbmtlZF9wYXJlbnRzOiBzZXRbaW50XSA9IHNldCgpCiAgICAgICAgcm93X2luZCwgY29sX2luZCA9IGxpbmVhcl9zdW1fYXNzaWdubWVudChjb3N0KQogICAgICAgIGZvciBpLCBqX2xvY2FsIGluIHppcChyb3dfaW5kLCBjb2xfaW5kKToKICAgICAgICAgICAgaWYgY29zdFtpLCBqX2xvY2FsXSA+PSAxZTU6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBqID0gY2hpbGRfaW5kaWNlc1tqX2xvY2FsXQogICAgICAgICAgICBzZWxmLmVkZ2VzLmFwcGVuZCgocGFyZW50c1tpXS5nbG9iYWxfaWQsIGNoaWxkX2dsb2JhbF9pZHNbal0pKQogICAgICAgICAgICBtYXRjaGVkX2NoaWxkcmVuLmFkZChqKQogICAgICAgICAgICBsaW5rZWRfcGFyZW50cy5hZGQocGFyZW50c1tpXS5nbG9iYWxfaWQpCiAgICAgICAgcmV0dXJuIG1hdGNoZWRfY2hpbGRyZW4sIGxpbmtlZF9wYXJlbnRzCgogICAgZGVmIF9taXRvc2lzX2xpbmtzKHNlbGYsIHBhcmVudDogT3BlblRyYWNrLCBjaGlsZF9pbmRpY2VzOiBsaXN0W2ludF0sCiAgICAgICAgICAgICAgICAgICAgICAgY2hpbGRyZW46IGxpc3RbZGljdF0sIGNoaWxkX2dsb2JhbF9pZHM6IGxpc3RbaW50XSkgLT4gbGlzdFtpbnRdOgogICAgICAgIGlmIG5vdCBtaXRvc2lzX3JlYWR5KHBhcmVudC5jZWxsLmdldCgicGh5c2ljYWxfdm9sdW1lIiwgcGFyZW50LmNlbGwuZ2V0KCJ2b2wiLCAwLjApKSk6CiAgICAgICAgICAgIHJldHVybiBbXQogICAgICAgIHJldHVybiBzdXBlcigpLl9taXRvc2lzX2xpbmtzKHBhcmVudCwgY2hpbGRfaW5kaWNlcywgY2hpbGRyZW4sIGNoaWxkX2dsb2JhbF9pZHMpCgoKZGVmIHJ1bl90cmFja2luZygKICAgIGZyYW1lczogbGlzdFtsaXN0W2RpY3RdXSwKICAgIGdsb2JhbF9pZHNfcGVyX2ZyYW1lOiBsaXN0W2xpc3RbaW50XV0sCikgLT4gbGlzdFt0dXBsZVtpbnQsIGludF1dOgogICAgdHJhY2tlciA9IENvbXBldGl0aXZlVHJhY2tlcigpCiAgICBmb3IgdCwgKGNlbGxzLCBnaWRzKSBpbiBlbnVtZXJhdGUoemlwKGZyYW1lcywgZ2xvYmFsX2lkc19wZXJfZnJhbWUpKToKICAgICAgICB0cmFja2VyLmFkdmFuY2UodCwgY2VsbHMsIGdpZHMpCiAgICByZXR1cm4gbGlzdCh0cmFja2VyLmVkZ2VzKQoKCmRlZiBtYWtlX2JlbmNobWFya190cmFja2VyKCk6CiAgICAiIiJSZXR1cm4gYSB0cmFja3NkYXRhLWNvbXBhdGlibGUgdHJhY2tlciBmdW5jdGlvbi4iIiIKICAgIGRlZiBfdHJhY2soZnJhbWVzKToKICAgICAgICBpbXBvcnQgcG9sYXJzIGFzIHBsCiAgICAgICAgaW1wb3J0IHRyYWNrc2RhdGEgYXMgdGQKICAgICAgICBmcm9tIGZzb3RfY2VsbHVsYXJfYnJpZGdlIGltcG9ydCB0cmFja19zZXF1ZW5jZQoKICAgICAgICBnLCBwZXJfdCA9IGZyYW1lc1siZ3JhcGgiXSwgZnJhbWVzWyJwZXJfdCJdCiAgICAgICAgbm9ybV9wZXJfdCA9IFtdCiAgICAgICAgZm9yIGNlbGxzLCBpZHMgaW4gcGVyX3Q6CiAgICAgICAgICAgIG5vcm1fY2VsbHMgPSBbXQogICAgICAgICAgICBmb3IgYyBpbiBjZWxsczoKICAgICAgICAgICAgICAgIG5jID0gZGljdChjKQogICAgICAgICAgICAgICAgbmNbInBoeXNpY2FsX3ZvbHVtZSJdID0gYy5nZXQoInZvbCIsIGMuZ2V0KCJwaHlzaWNhbF92b2x1bWUiLCAwLjApKQogICAgICAgICAgICAgICAgbm9ybV9jZWxscy5hcHBlbmQobmMpCiAgICAgICAgICAgIG5vcm1fcGVyX3QuYXBwZW5kKChub3JtX2NlbGxzLCBpZHMpKQoKICAgICAgICAjIFVzZSBDb21wZXRpdGl2ZVRyYWNrZXIgdmlhIHBhdGNoZWQgc2VxdWVuY2Ug4oCUIHJldXNlIGJyaWRnZSB0cmFja19zZXF1ZW5jZQogICAgICAgICMgYnV0IHN3YXAgaW4gY29tcGV0aXRpdmUgYXNzaWdubWVudCB0aHJvdWdoIHJ1bl90cmFja2luZyBvbiBpbmRleCBpZHMuCiAgICAgICAgdHJhY2tlciA9IENvbXBldGl0aXZlVHJhY2tlcigpCiAgICAgICAgaWRfdG9fbG9jOiBkaWN0W2ludCwgdHVwbGVbaW50LCBpbnRdXSA9IHt9CiAgICAgICAgZWRnZV9xdWFkcyA9IFtdCiAgICAgICAgcHJldiA9IDAKICAgICAgICBmb3IgdCwgKGNlbGxzLCBpZHMpIGluIGVudW1lcmF0ZShub3JtX3Blcl90KToKICAgICAgICAgICAgZm9yIGosIG5pZCBpbiBlbnVtZXJhdGUoaWRzKToKICAgICAgICAgICAgICAgIGlkX3RvX2xvY1tuaWRdID0gKHQsIGopCiAgICAgICAgICAgIHRyYWNrZXIuYWR2YW5jZSh0LCBjZWxscywgaWRzKQogICAgICAgICAgICBmb3Igc3JjX2dpZCwgdGd0X2dpZCBpbiB0cmFja2VyLmVkZ2VzW3ByZXY6XToKICAgICAgICAgICAgICAgIGlmIHNyY19naWQgaW4gaWRfdG9fbG9jIGFuZCB0Z3RfZ2lkIGluIGlkX3RvX2xvYzoKICAgICAgICAgICAgICAgICAgICB0cywgaXNyYyA9IGlkX3RvX2xvY1tzcmNfZ2lkXQogICAgICAgICAgICAgICAgICAgIHR0LCBqdGd0ID0gaWRfdG9fbG9jW3RndF9naWRdCiAgICAgICAgICAgICAgICAgICAgZWRnZV9xdWFkcy5hcHBlbmQoKHRzLCBpc3JjLCB0dCwganRndCkpCiAgICAgICAgICAgIHByZXYgPSBsZW4odHJhY2tlci5lZGdlcykKCiAgICAgICAgZm9yIHRzLCBpc3JjLCB0dCwganRndCBpbiBlZGdlX3F1YWRzOgogICAgICAgICAgICBnLmFkZF9lZGdlKG5vcm1fcGVyX3RbdHNdWzFdW2lzcmNdLCBub3JtX3Blcl90W3R0XVsxXVtqdGd0XSwge30pCiAgICAgICAgcmV0dXJuIGcKCiAgICByZXR1cm4gX3RyYWNr"
with open('biohub_competitive.py', 'wb') as _f:
    _f.write(base64.b64decode(_B64))
print('biohub_competitive.py written to working dir.')

In [ ]:
# Write fsot_original_competition.py to the working dir (offline Kaggle bundle).
import base64
_B64 = "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiIKRlNPVCBPcmlnaW5hbCBQcm9ncmFtIOKGkiBLYWdnbGUgQ29tcGV0aXRpb24gUG9ydAo9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KTWFwcyB0aGUgcHJlLWNvbXBldGl0aW9uIGNvZGViYXNlIChmc290X3JuYV90cmluYXJ5X2V2b2x1dGlvbl9zaW0pIHRvIEJpb2h1YiBzdWJtaXNzaW9uLgoKT3JpZ2luYWwgcHJvZ3JhbSBjaGFpbiAoc2FtZSByZXBvLCBmaWxlcy03ZWQxZTYyYyk6CiAgemFycl9pbmdlc3Rpb25fcGlwZWxpbmUucHkgICAgIExvYWQgT01FLVphcnIgKFQsWixZLFgpCiAgZnNvdF9mdWxsX3BpcGVsaW5lX3Rlc3QucHkgICAgIFNjaVB5L3RocmVzaG9sZCB2aXNpb24gcHJveHkgKyBmc290X3RyYWNraW5nX2VuZ2luZQogIGZzb3RfbWl0b3Npc19wcmVkaWN0b3IucHkgICAgICBQaGFzZS1zY2FsYXIgUyBtaXRvc2lzIGdhdGVzCiAgZnNvdF8zZF92aXNpb25fbmV0d29yay5weSAgICAgIEZTT1QtTk4gUGhhc2VHYXRlIGRldGVjdG9yIChmdXR1cmUgbmF0aXZlIHZpc2lvbikKICBmc290X3VuZXRfZ2F0ZXdheS5weSAgICAgICAgICAgVmlzaW9uIGNlbnRyb2lkcyDihpIgRlNPVCBsaW5lYWdlIHJlc29sdmVyCiAgZnNvdF9jZWxsdWxhcl9icmlkZ2UucHkgICAgICAgIFByb2R1Y3Rpb24gdHJhY2tlcjogbGlua19jb3N0ICsgbWl0b3Npc19yZWFkeSAoZnNvdF9jb3JlKQogIGthZ2dsZV9wcm90b3R5cGVfZnNvdF90cmFja2VyLnB5ICBzdWJtaXNzaW9uLmNzdiBub2RlL2VkZ2UgZm9ybWF0CgpDb21wZXRpdGlvbiBwb3J0ICh0aGlzIG1vZHVsZSk6CiAgZGV0ZWN0ICDihpIgQ29tcGV0aXRpdmVEZXRlY3RvciAocGVha3MvY2VsbHBvc2UpIE9SIFUtTmV0IGNvb3JkcyAoZ2F0ZXdheSBtb2RlKQogIGxpbmsgICAg4oaSIGZzb3RfY2VsbHVsYXJfYnJpZGdlLlNlcXVlbmNlVHJhY2tlciAoMTAwJSBGU09UIHNjYWxhciBtYXRoKQogIGV4cG9ydCAg4oaSIHBlci1kYXRhc2V0IG5vZGVfaWQgcmVzZXQgKyB2YWxpZGF0aW9uCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IG9zCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IGRhc2suYXJyYXkgYXMgZGEKaW1wb3J0IHphcnIKCmZyb20gYmlvaHViX2NvbXBldGl0aXZlIGltcG9ydCBDb21wZXRpdGl2ZURldGVjdG9yLCBEYXRhc2V0Q29udGV4dApmcm9tIGZzb3RfY2VsbHVsYXJfYnJpZGdlIGltcG9ydCBTZXF1ZW5jZVRyYWNrZXIKZnJvbSBmc290X2NvcmUgaW1wb3J0IENPTExBUFNFX1RIUkVTSE9MRCwgY29tcHV0ZV9zY2FsYXJfYmlvbG9naWNhbAoKUk9PVCA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnQKT1JJR0lOID0gImZzb3Rfcm5hX3RyaW5hcnlfZXZvbHV0aW9uX3NpbS9maWxlcy03ZWQxZTYyYyIKCgpkZWYgdmVyaWZ5X2Zzb3RfY29yZSgpIC0+IE5vbmU6CiAgICAiIiJQcm92ZSBmc290X2NvcmUgc2NhbGFyIGVuZ2luZSBpcyBsaXZlIGJlZm9yZSB0cmFja2luZy4iIiIKICAgIHMgPSBjb21wdXRlX3NjYWxhcl9iaW9sb2dpY2FsKGRlbHRhX3BzaT0wLjM1LCBvYnNlcnZlZD1UcnVlKQogICAgcHJpbnQoZiJbRlNPVC1DT1JFXSBiaW9sb2dpY2FsIHNjYWxhciBTPXtzOi42Zn0gIGNvbGxhcHNlX3RocmVzaG9sZD17Q09MTEFQU0VfVEhSRVNIT0xEOi40Zn0iKQogICAgcHJpbnQoZiJbRlNPVC1DT1JFXSBMZWFuIHJlZjogZ2l0aHViLmNvbS9kYXBwYWx1bWJvOTEvRlNPVC0yLjEtTGVhbiIpCiAgICBwcmludChmIltGU09ULUNPUkVdIG9yaWdpbjoge09SSUdJTn0iKQoKCmRlZiBfb3Blbl92aWRlbyh6YXJyX3BhdGg6IHN0ciB8IFBhdGgpIC0+IHR1cGxlW2RhLkFycmF5LCBEYXRhc2V0Q29udGV4dF06CiAgICAiIiJMb2FkIGxhenkgdmlkZW8gYXJyYXkgKyBxdWFudGlsZSBjb250ZXh0ICh6YXJyX2luZ2VzdGlvbl9waXBlbGluZSBwYXR0ZXJuKS4iIiIKICAgIHphcnJfcGF0aCA9IFBhdGgoemFycl9wYXRoKQogICAgc3RvcmUgPSB6YXJyX3BhdGggaWYgemFycl9wYXRoLnN1ZmZpeCA9PSAiLnphcnIiIGVsc2UgemFycl9wYXRoLnBhcmVudCAvIGYie3phcnJfcGF0aC5uYW1lfS56YXJyIgogICAgdHJ5OgogICAgICAgIGF0dHJzID0gZGljdCh6YXJyLm9wZW5fZ3JvdXAoc3RyKHN0b3JlKSwgbW9kZT0iciIpLmF0dHJzKQogICAgICAgIGN0eCA9IERhdGFzZXRDb250ZXh0KAogICAgICAgICAgICBxdWFudGlsZXM9YXR0cnMuZ2V0KCJpbWFnZV9zdGF0aXN0aWNzIiwge30pLmdldCgicXVhbnRpbGVzIiwge30pLAogICAgICAgICkKICAgIGV4Y2VwdCBFeGNlcHRpb246ICAjIG5vcWE6IEJMRTAwMQogICAgICAgIGN0eCA9IERhdGFzZXRDb250ZXh0KCkKICAgIHZpZGVvID0gZGEuZnJvbV96YXJyKHphcnIub3BlbihzdHIoc3RvcmUpLCBtb2RlPSJyIilbIjAiXSkKICAgIGlmIHZpZGVvLm5kaW0gPT0gNToKICAgICAgICB2aWRlbyA9IHZpZGVvWzosIDAsIDosIDosIDpdCiAgICByZXR1cm4gdmlkZW8sIGN0eAoKCmRlZiB0cmFja19kYXRhc2V0X25hdGl2ZSgKICAgIHphcnJfcGF0aDogc3RyIHwgUGF0aCwKICAgIGRhdGFzZXRfbmFtZTogc3RyLAogICAgZGV0ZWN0b3JfbW9kZTogc3RyIHwgTm9uZSA9IE5vbmUsCiAgICByb3dfc3RhcnQ6IGludCA9IDAsCikgLT4gdHVwbGVbbGlzdFtkaWN0XSwgaW50XToKICAgICIiIgogICAgT3JpZ2luYWwgRlNPVCBwcm9ncmFtIHBvcnQ6IHZpc2lvbiBkZXRlY3Qg4oaSIFNlcXVlbmNlVHJhY2tlciDihpIgc3VibWlzc2lvbiByb3dzLgoKICAgIEV2b2x2ZXMgZnNvdF9mdWxsX3BpcGVsaW5lX3Rlc3QuZnNvdF90cmFja2luZ19lbmdpbmUgaW50byBmdWxsLXZpZGVvCiAgICBmc290X2NlbGx1bGFyX2JyaWRnZS5TZXF1ZW5jZVRyYWNrZXIgd2l0aCBjb21wZXRpdGlvbiBxdWFudGlsZSBwZWFrcy9jZWxscG9zZS4KICAgICIiIgogICAgbW9kZSA9IGRldGVjdG9yX21vZGUgb3Igb3MuZW52aXJvbi5nZXQoIkJJT0hVQl9ERVRFQ1RPUiIsICJwZWFrcyIpCiAgICBkZXQgPSBDb21wZXRpdGl2ZURldGVjdG9yKG1vZGU9bW9kZSkKICAgIHZpZGVvLCBjdHggPSBfb3Blbl92aWRlbyh6YXJyX3BhdGgpCiAgICBkZXQuY3R4ID0gY3R4CgogICAgcm93czogbGlzdFtkaWN0XSA9IFtdCiAgICByb3dfaWR4ID0gcm93X3N0YXJ0CiAgICBub2RlX2lkID0gMQogICAgdHJhY2tlciA9IFNlcXVlbmNlVHJhY2tlcigpCiAgICBwcmV2X2VkZ2VzID0gMAoKICAgIHByaW50KGYiW0ZTT1QtTkFUSVZFXSB7ZGF0YXNldF9uYW1lfSBkZXRlY3Rvcj17bW9kZX0gZnJhbWVzPXt2aWRlby5zaGFwZVswXX0iKQogICAgZm9yIHQgaW4gcmFuZ2UodmlkZW8uc2hhcGVbMF0pOgogICAgICAgIGNlbGxzID0gZGV0KHZpZGVvW3RdLmNvbXB1dGUoKSkKICAgICAgICBnaWRzOiBsaXN0W2ludF0gPSBbXQogICAgICAgIGZvciBjIGluIGNlbGxzOgogICAgICAgICAgICBnaWRzLmFwcGVuZChub2RlX2lkKQogICAgICAgICAgICByb3dzLmFwcGVuZCh7CiAgICAgICAgICAgICAgICAiaWQiOiByb3dfaWR4LCAiZGF0YXNldCI6IGRhdGFzZXRfbmFtZSwgInJvd190eXBlIjogIm5vZGUiLAogICAgICAgICAgICAgICAgIm5vZGVfaWQiOiBub2RlX2lkLCAidCI6IHQsCiAgICAgICAgICAgICAgICAieiI6IGludChyb3VuZChjWyJ6Il0pKSwgInkiOiBpbnQocm91bmQoY1sieSJdKSksICJ4IjogaW50KHJvdW5kKGNbIngiXSkpLAogICAgICAgICAgICAgICAgInNvdXJjZV9pZCI6IC0xLCAidGFyZ2V0X2lkIjogLTEsCiAgICAgICAgICAgIH0pCiAgICAgICAgICAgIHJvd19pZHggKz0gMQogICAgICAgICAgICBub2RlX2lkICs9IDEKICAgICAgICB0cmFja2VyLmFkdmFuY2UodCwgY2VsbHMsIGdpZHMpCiAgICAgICAgZm9yIHNyYywgdGd0IGluIHRyYWNrZXIuZWRnZXNbcHJldl9lZGdlczpdOgogICAgICAgICAgICByb3dzLmFwcGVuZCh7CiAgICAgICAgICAgICAgICAiaWQiOiByb3dfaWR4LCAiZGF0YXNldCI6IGRhdGFzZXRfbmFtZSwgInJvd190eXBlIjogImVkZ2UiLAogICAgICAgICAgICAgICAgIm5vZGVfaWQiOiAtMSwgInQiOiAtMSwgInoiOiAtMSwgInkiOiAtMSwgIngiOiAtMSwKICAgICAgICAgICAgICAgICJzb3VyY2VfaWQiOiBzcmMsICJ0YXJnZXRfaWQiOiB0Z3QsCiAgICAgICAgICAgIH0pCiAgICAgICAgICAgIHJvd19pZHggKz0gMQogICAgICAgIHByZXZfZWRnZXMgPSBsZW4odHJhY2tlci5lZGdlcykKCiAgICBwcmludChmIltGU09ULU5BVElWRV0ge2RhdGFzZXRfbmFtZX0gbm9kZXM9e25vZGVfaWQgLSAxfSBlZGdlcz17bGVuKHRyYWNrZXIuZWRnZXMpfSIpCiAgICBwcmludChkZXQuc3RhdHMuc3VtbWFyeSgpKQogICAgcmV0dXJuIHJvd3MsIHJvd19pZHgKCgpkZWYgdHJhY2tfYWxsX2RhdGFzZXRzKAogICAgZGF0YV9kaXI6IHN0ciB8IFBhdGgsCiAgICBkZXRlY3Rvcl9tb2RlOiBzdHIgfCBOb25lID0gTm9uZSwKKSAtPiBsaXN0W2RpY3RdOgogICAgIiIiUnVuIG5hdGl2ZSBGU09UIHBvcnQgb24gZXZlcnkgLnphcnIgaW4gYSBjb21wZXRpdGlvbiBkYXRhIGRpcmVjdG9yeS4iIiIKICAgIHZlcmlmeV9mc290X2NvcmUoKQogICAgZGF0YV9kaXIgPSBQYXRoKGRhdGFfZGlyKQogICAgcm93czogbGlzdFtkaWN0XSA9IFtdCiAgICByb3dfaWR4ID0gMAogICAgZm9yIGRzX25hbWUgaW4gc29ydGVkKGQgZm9yIGQgaW4gb3MubGlzdGRpcihkYXRhX2RpcikgaWYgZC5lbmRzd2l0aCgiLnphcnIiKSk6CiAgICAgICAgY2xlYW4gPSBkc19uYW1lLnJlcGxhY2UoIi56YXJyIiwgIiIpCiAgICAgICAgcGFydCwgcm93X2lkeCA9IHRyYWNrX2RhdGFzZXRfbmF0aXZlKAogICAgICAgICAgICBkYXRhX2RpciAvIGRzX25hbWUsIGNsZWFuLCBkZXRlY3Rvcl9tb2RlPWRldGVjdG9yX21vZGUsIHJvd19zdGFydD1yb3dfaWR4LAogICAgICAgICkKICAgICAgICByb3dzLmV4dGVuZChwYXJ0KQogICAgcmV0dXJuIHJvd3M="
with open('fsot_original_competition.py', 'wb') as _f:
    _f.write(base64.b64decode(_B64))
print('fsot_original_competition.py written to working dir.')

In [ ]:
# Write biohub_unet_engine.py to the working dir (offline Kaggle bundle).
import base64
_B64 = "..."
with open('biohub_unet_engine.py', 'wb') as _f:
    _f.write(base64.b64decode(_B64))
print('biohub_unet_engine.py written to working dir.')

In [ ]:
# Write validate_kaggle_submission.py to the working dir (offline Kaggle bundle).
import base64
_B64 = "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJQcmUtZmxpZ2h0IHZhbGlkYXRpb24gbWlycm9yaW5nIEthZ2dsZSdzIGNzdl90b19nZWZmcyArIGNvbXBldGl0aW9uIHJ1bGVzLiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBzeXMKaW1wb3J0IHRlbXBmaWxlCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IHBvbGFycyBhcyBwbAoKUk9PVCA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnQKX1JFUE8gPSBST09UIC8gImthZ2dsZS1jZWxsLXRyYWNraW5nLWNvbXBldGl0aW9uIgpmb3IgcCBpbiAoCiAgICBST09ULAogICAgX1JFUE8gLyAic3JjIiwKICAgIF9SRVBPIC8gInNjcmlwdHMiLAogICAgUGF0aCgiL2thZ2dsZS93b3JraW5nIiksCiAgICBQYXRoKCIva2FnZ2xlL3dvcmtpbmcvY2VsbG1vdF9idW5kbGUvc2NyaXB0cyIpLAopOgogICAgaWYgcC5leGlzdHMoKToKICAgICAgICBzeXMucGF0aC5pbnNlcnQoMCwgc3RyKHApKQoKZnJvbSBjc3ZfdG9fZ2VmZnMgaW1wb3J0IGJ1aWxkX2dyYXBoX2Zyb21fcm93cywgY3N2X3RvX2dlZmZzICAjIG5vcWE6IEU0MDIKCkNPTFVNTlMgPSAoCiAgICAiaWQiLCAiZGF0YXNldCIsICJyb3dfdHlwZSIsICJub2RlX2lkIiwgInQiLCAieiIsICJ5IiwgIngiLCAic291cmNlX2lkIiwgInRhcmdldF9pZCIsCikKVEVTVF9EQVRBU0VUUyA9ICgKICAgICI0NGI2XzAxMTNkZTNiIiwKICAgICI0NGI2XzBiMjQ4NDVmIiwKICAgICI2YmJhXzA1YjY4NTBiIiwKICAgICI2YmJhXzA1ZGIwZmIxIiwKKQoKCmRlZiB2YWxpZGF0ZV9jc3YoY3N2X3BhdGg6IFBhdGgsIHN0cmljdF9kYXRhc2V0czogYm9vbCA9IFRydWUpIC0+IGxpc3Rbc3RyXToKICAgIGVycm9yczogbGlzdFtzdHJdID0gW10KICAgIGRmID0gcGwucmVhZF9jc3YoY3N2X3BhdGgpCgogICAgbWlzc2luZyA9IFtjIGZvciBjIGluIENPTFVNTlMgaWYgYyBub3QgaW4gZGYuY29sdW1uc10KICAgIGlmIG1pc3Npbmc6CiAgICAgICAgZXJyb3JzLmFwcGVuZChmIm1pc3NpbmcgY29sdW1uczoge21pc3Npbmd9IikKICAgICAgICByZXR1cm4gZXJyb3JzCgogICAgaWYgZGZbImlkIl0ubnVsbF9jb3VudCgpOgogICAgICAgIGVycm9ycy5hcHBlbmQoIm51bGwgdmFsdWVzIGluIGlkIGNvbHVtbiIpCiAgICBpZiBkZlsiaWQiXS5uX3VuaXF1ZSgpICE9IGRmLmhlaWdodDoKICAgICAgICBlcnJvcnMuYXBwZW5kKCJpZCBjb2x1bW4gbXVzdCBiZSB1bmlxdWUiKQoKICAgIGRhdGFzZXRzID0gc29ydGVkKGRmWyJkYXRhc2V0Il0udW5pcXVlKCkudG9fbGlzdCgpKQogICAgaWYgc3RyaWN0X2RhdGFzZXRzIGFuZCBkYXRhc2V0cyAhPSBsaXN0KFRFU1RfREFUQVNFVFMpOgogICAgICAgIGVycm9ycy5hcHBlbmQoZiJleHBlY3RlZCBkYXRhc2V0cyB7VEVTVF9EQVRBU0VUU30sIGdvdCB7ZGF0YXNldHN9IikKCiAgICBmb3IgbmFtZSBpbiBkYXRhc2V0czoKICAgICAgICBncm91cCA9IGRmLmZpbHRlcihwbC5jb2woImRhdGFzZXQiKSA9PSBuYW1lKQogICAgICAgIG5vZGVzID0gZ3JvdXAuZmlsdGVyKHBsLmNvbCgicm93X3R5cGUiKSA9PSAibm9kZSIpCiAgICAgICAgZWRnZXMgPSBncm91cC5maWx0ZXIocGwuY29sKCJyb3dfdHlwZSIpID09ICJlZGdlIikKCiAgICAgICAgaWYgbm9kZXMuaGVpZ2h0ID09IDA6CiAgICAgICAgICAgIGVycm9ycy5hcHBlbmQoZiJ7bmFtZX06IG5vIG5vZGUgcm93cyIpCiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgaWYgZWRnZXMuaGVpZ2h0ID09IDA6CiAgICAgICAgICAgIGVycm9ycy5hcHBlbmQoZiJ7bmFtZX06IG5vIGVkZ2Ugcm93cyIpCgogICAgICAgIG5vZGVfaWRzID0gbm9kZXNbIm5vZGVfaWQiXS50b19saXN0KCkKICAgICAgICBpZiBtaW4obm9kZV9pZHMpICE9IDE6CiAgICAgICAgICAgIGVycm9ycy5hcHBlbmQoZiJ7bmFtZX06IG5vZGVfaWQgbXVzdCBzdGFydCBhdCAxLCBnb3QgbWluPXttaW4obm9kZV9pZHMpfSIpCiAgICAgICAgaWYgbGVuKHNldChub2RlX2lkcykpICE9IGxlbihub2RlX2lkcyk6CiAgICAgICAgICAgIGVycm9ycy5hcHBlbmQoZiJ7bmFtZX06IGR1cGxpY2F0ZSBub2RlX2lkIHZhbHVlcyBpbiBub2RlIHJvd3MiKQoKICAgICAgICBiYWRfbm9kZXMgPSBub2Rlcy5maWx0ZXIoCiAgICAgICAgICAgIChwbC5jb2woInNvdXJjZV9pZCIpICE9IC0xKSB8IChwbC5jb2woInRhcmdldF9pZCIpICE9IC0xKQogICAgICAgICkKICAgICAgICBpZiBiYWRfbm9kZXMuaGVpZ2h0OgogICAgICAgICAgICBlcnJvcnMuYXBwZW5kKGYie25hbWV9OiB7YmFkX25vZGVzLmhlaWdodH0gbm9kZSByb3dzIGhhdmUgbm9uIC0xIHNvdXJjZS90YXJnZXQiKQoKICAgICAgICBiYWRfZWRnZXMgPSBlZGdlcy5maWx0ZXIoCiAgICAgICAgICAgIChwbC5jb2woIm5vZGVfaWQiKSAhPSAtMSkKICAgICAgICAgICAgfCAocGwuY29sKCJ0IikgIT0gLTEpCiAgICAgICAgICAgIHwgKHBsLmNvbCgieiIpICE9IC0xKQogICAgICAgICAgICB8IChwbC5jb2woInkiKSAhPSAtMSkKICAgICAgICAgICAgfCAocGwuY29sKCJ4IikgIT0gLTEpCiAgICAgICAgKQogICAgICAgIGlmIGJhZF9lZGdlcy5oZWlnaHQ6CiAgICAgICAgICAgIGVycm9ycy5hcHBlbmQoZiJ7bmFtZX06IHtiYWRfZWRnZXMuaGVpZ2h0fSBlZGdlIHJvd3MgaGF2ZSBpbnZhbGlkIHBsYWNlaG9sZGVyIGZpZWxkcyIpCgogICAgICAgIG5vZGVfc2V0ID0gc2V0KG5vZGVfaWRzKQogICAgICAgIGZvciBjb2wgaW4gKCJzb3VyY2VfaWQiLCAidGFyZ2V0X2lkIik6CiAgICAgICAgICAgIGJhZCA9IGVkZ2VzLmZpbHRlcih+cGwuY29sKGNvbCkuaXNfaW4obGlzdChub2RlX3NldCkpKQogICAgICAgICAgICBpZiBiYWQuaGVpZ2h0OgogICAgICAgICAgICAgICAgZXJyb3JzLmFwcGVuZChmIntuYW1lfToge2JhZC5oZWlnaHR9IGVkZ2VzIHJlZmVyZW5jZSBtaXNzaW5nIHtjb2x9IikKCiAgICAgICAgZHVwX2VkZ2VzID0gZWRnZXMuZ3JvdXBfYnkoWyJzb3VyY2VfaWQiLCAidGFyZ2V0X2lkIl0pLmxlbigpLmZpbHRlcihwbC5jb2woImxlbiIpID4gMSkKICAgICAgICBpZiBkdXBfZWRnZXMuaGVpZ2h0OgogICAgICAgICAgICBlcnJvcnMuYXBwZW5kKGYie25hbWV9OiB7ZHVwX2VkZ2VzLmhlaWdodH0gZHVwbGljYXRlIChzb3VyY2VfaWQsIHRhcmdldF9pZCkgcGFpcnMiKQoKICAgICAgICAjIE9mZmljaWFsIGxvYWRlciBtdXN0IHN1Y2NlZWQgKHN0cmljdCBub2RlX2lkIHppcCkuCiAgICAgICAgdHJ5OgogICAgICAgICAgICBidWlsZF9ncmFwaF9mcm9tX3Jvd3Mobm9kZXMsIGVkZ2VzKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOiAgIyBub3FhOiBCTEUwMDEKICAgICAgICAgICAgZXJyb3JzLmFwcGVuZChmIntuYW1lfTogY3N2X3RvX2dlZmZzIGJ1aWxkIGZhaWxlZDoge2V4Y30iKQoKICAgICAgICB0X21heCA9IGludChub2Rlc1sidCJdLm1heCgpKQogICAgICAgIHRfbWluID0gaW50KG5vZGVzWyJ0Il0ubWluKCkpCiAgICAgICAgaWYgdF9taW4gPCAwOgogICAgICAgICAgICBlcnJvcnMuYXBwZW5kKGYie25hbWV9OiBub2RlIHQgPCAwIikKICAgICAgICBpZiB0X21heCA8IDk5OgogICAgICAgICAgICBlcnJvcnMuYXBwZW5kKGYie25hbWV9OiBvbmx5IGNvdmVycyB0PTAuLnt0X21heH0sIGV4cGVjdGVkIDAuLjk5IikKCiAgICByZXR1cm4gZXJyb3JzCgoKZGVmIG1haW4oKSAtPiBOb25lOgogICAgYXAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj0iVmFsaWRhdGUgc3VibWlzc2lvbi5jc3YgYmVmb3JlIEthZ2dsZSB1cGxvYWQiKQogICAgYXAuYWRkX2FyZ3VtZW50KCJjc3YiLCB0eXBlPVBhdGgpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tc2NvcmUiLCBhY3Rpb249InN0b3JlX3RydWUiLCBoZWxwPSJBbHNvIHNjb3JlIGFnYWluc3QgdHJhaW4gR1QiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWd0LWRpciIsIHR5cGU9UGF0aCwgZGVmYXVsdD1QYXRoKHIiRDpcS2FnZ2xlX0Jpb2h1Yl9EYXRhXHRyYWluIikpCiAgICBhcmdzID0gYXAucGFyc2VfYXJncygpCgogICAgZXJyb3JzID0gdmFsaWRhdGVfY3N2KGFyZ3MuY3N2KQogICAgaWYgZXJyb3JzOgogICAgICAgIHByaW50KCJJTlZBTElEIHN1Ym1pc3Npb246IikKICAgICAgICBmb3IgZXJyIGluIGVycm9yczoKICAgICAgICAgICAgcHJpbnQoZiIgIC0ge2Vycn0iKQogICAgICAgIHJhaXNlIFN5c3RlbUV4aXQoMSkKCiAgICBwcmludChmIlZBTElEOiB7YXJncy5jc3Z9ICh7cGwucmVhZF9jc3YoYXJncy5jc3YpLmhlaWdodH0gcm93cykiKQoKICAgIHdpdGggdGVtcGZpbGUuVGVtcG9yYXJ5RGlyZWN0b3J5KHByZWZpeD0iYmlvaHViX3ZhbGlkYXRlXyIpIGFzIHRtcDoKICAgICAgICBjc3ZfdG9fZ2VmZnMoYXJncy5jc3YsIFBhdGgodG1wKSkKICAgICAgICBwcmludChmImNzdl90b19nZWZmcyByb3VuZC10cmlwIE9LICh7bGVuKGxpc3QoUGF0aCh0bXApLmdsb2IoJyouZ2VmZicpKSl9IGdlZmZzKSIpCgogICAgaWYgYXJncy5zY29yZToKICAgICAgICBmcm9tIGthZ2dsZV9zdWJtaXNzaW9uX3Njb3JlIGltcG9ydCBzY29yZV9jc3YgICMgbm9xYTogV1BTNDMzCgogICAgICAgIG91dCA9IHNjb3JlX2NzdihhcmdzLmNzdiwgYXJncy5ndF9kaXIpCiAgICAgICAgcHJpbnQoZiJUcmFpbiBwcm94eSBzY29yZToge291dFsnc2NvcmUnXTouNGZ9IChhZGo9e291dFsnYWRqX2VkZ2VfamFjY2FyZCddOi40Zn0pIikKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigp"
with open('validate_kaggle_submission.py', 'wb') as _f:
    _f.write(base64.b64decode(_B64))
print('validate_kaggle_submission.py written to working dir.')

In [ ]:
# Write csv_to_geffs.py to the working dir (offline Kaggle bundle).
import base64
_B64 = "ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IHBvbGFycyBhcyBwbAppbXBvcnQgdHJhY2tzZGF0YSBhcyB0ZAoKZnJvbSB0cmFja2luZ19jZWxsbW90LmlvIGltcG9ydCBzYXZlX2dyYXBoCgoKZGVmIGJ1aWxkX2dyYXBoX2Zyb21fcm93cygKICAgIG5vZGVfcm93czogcGwuRGF0YUZyYW1lLAogICAgZWRnZV9yb3dzOiBwbC5EYXRhRnJhbWUsCikgLT4gdGQuZ3JhcGguSW5NZW1vcnlHcmFwaDoKICAgICIiIlJlYnVpbGQgYSB0cmFja3NkYXRhIGdyYXBoIGZyb20gb25lIGRhdGFzZXQncyBub2RlIGFuZCBlZGdlIHJvd3MuCgogICAgdHJhY2tzZGF0YSBhc3NpZ25zIGZyZXNoIG5vZGUgaWRzLCBzbyBDU1YgYGBub2RlX2lkYGBzIGFyZSByZW1hcHBlZCB3aGVuCiAgICBhZGRpbmcgZWRnZXMgKG1hdGNoaW5nIGlzIHNwYXRpYWwsIG5vdCBieSBpZCkuCiAgICAiIiIKICAgIGdyYXBoID0gdGQuZ3JhcGguSW5NZW1vcnlHcmFwaCgpCiAgICBmb3Iga2V5IGluICgieiIsICJ5IiwgIngiKToKICAgICAgICBncmFwaC5hZGRfbm9kZV9hdHRyX2tleShrZXksIHBsLkZsb2F0NjQsIC05OTk5OTkuMCkKCiAgICBhc3NpZ25lZCA9IGdyYXBoLmJ1bGtfYWRkX25vZGVzKAogICAgICAgIG5vZGVfcm93cy5zZWxlY3QoCiAgICAgICAgICAgIHBsLmNvbCgidCIpLmNhc3QocGwuSW50NjQpLAogICAgICAgICAgICBwbC5jb2woInoiKS5jYXN0KHBsLkZsb2F0NjQpLAogICAgICAgICAgICBwbC5jb2woInkiKS5jYXN0KHBsLkZsb2F0NjQpLAogICAgICAgICAgICBwbC5jb2woIngiKS5jYXN0KHBsLkZsb2F0NjQpLAogICAgICAgICkudG9fZGljdHMoKQogICAgKQogICAgaWRfbWFwID0gZGljdCh6aXAobm9kZV9yb3dzWyJub2RlX2lkIl0udG9fbGlzdCgpLCBhc3NpZ25lZCwgc3RyaWN0PVRydWUpKQoKICAgIGlmIGVkZ2Vfcm93cy5oZWlnaHQ6CiAgICAgICAgZ3JhcGguYnVsa19hZGRfZWRnZXMoCiAgICAgICAgICAgIFsKICAgICAgICAgICAgICAgIHsic291cmNlX2lkIjogaWRfbWFwW3NdLCAidGFyZ2V0X2lkIjogaWRfbWFwW3RdfQogICAgICAgICAgICAgICAgZm9yIHMsIHQgaW4gemlwKAogICAgICAgICAgICAgICAgICAgIGVkZ2Vfcm93c1sic291cmNlX2lkIl0udG9fbGlzdCgpLCBlZGdlX3Jvd3NbInRhcmdldF9pZCJdLnRvX2xpc3QoKSwgc3RyaWN0PVRydWUKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgXQogICAgICAgICkKCiAgICByZXR1cm4gZ3JhcGgKCgpkZWYgY3N2X3RvX2dlZmZzKAogICAgY3N2X3BhdGg6IFBhdGggfCBzdHIsCiAgICBvdXRfZGlyOiBQYXRoIHwgc3RyLAogICAgb3ZlcndyaXRlOiBib29sID0gVHJ1ZSwKKSAtPiBsaXN0W1BhdGhdOgogICAgIiIiQ29udmVydCBhIHN1Ym1pc3Npb24gQ1NWIGludG8gb25lIGBge2RhdGFzZXR9LmdlZmZgYCBpbiBgYG91dF9kaXJgYDsgcmV0dXJuIHRoZWlyIHBhdGhzLiIiIgogICAgY3N2X3BhdGggPSBQYXRoKGNzdl9wYXRoKQogICAgb3V0X2RpciA9IFBhdGgob3V0X2RpcikKICAgIG91dF9kaXIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQoKICAgIGRmID0gcGwucmVhZF9jc3YoCiAgICAgICAgY3N2X3BhdGgsCiAgICAgICAgY29sdW1ucz1bImRhdGFzZXQiLCAicm93X3R5cGUiLCAibm9kZV9pZCIsICJ0IiwgInoiLCAieSIsICJ4IiwgInNvdXJjZV9pZCIsICJ0YXJnZXRfaWQiXSwKICAgICkKCiAgICB3cml0dGVuOiBsaXN0W1BhdGhdID0gW10KICAgIGZvciAobmFtZSwpLCBncm91cCBpbiBkZi5ncm91cF9ieSgiZGF0YXNldCIpOgogICAgICAgIG5vZGVfcm93cyA9IGdyb3VwLmZpbHRlcihwbC5jb2woInJvd190eXBlIikgPT0gIm5vZGUiKQogICAgICAgIGVkZ2Vfcm93cyA9IGdyb3VwLmZpbHRlcihwbC5jb2woInJvd190eXBlIikgPT0gImVkZ2UiKQoKICAgICAgICBncmFwaCA9IGJ1aWxkX2dyYXBoX2Zyb21fcm93cyhub2RlX3Jvd3MsIGVkZ2Vfcm93cykKICAgICAgICBvdXRfcGF0aCA9IG91dF9kaXIgLyBmIntuYW1lfS5nZWZmIgogICAgICAgIHNhdmVfZ3JhcGgoZ3JhcGgsIG91dF9wYXRoLCBvdmVyd3JpdGU9b3ZlcndyaXRlKQogICAgICAgIHdyaXR0ZW4uYXBwZW5kKG91dF9wYXRoKQogICAgICAgIHByaW50KGYie25hbWV9OiB7bm9kZV9yb3dzLmhlaWdodH0gbm9kZXMsIHtlZGdlX3Jvd3MuaGVpZ2h0fSBlZGdlcyAtPiB7b3V0X3BhdGh9IikKCiAgICBwcmludChmIlxuV3JvdGUge2xlbih3cml0dGVuKX0gZ2VmZnMgdG8ge291dF9kaXJ9IikKICAgIHJldHVybiB3cml0dGVuCgoKZGVmIG1haW4oKSAtPiBOb25lOgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249IkNvbnZlcnQgYSBzdWJtaXNzaW9uIENTViBpbnRvIG9uZSAuZ2VmZiBwZXIgZGF0YXNldC4iKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1jc3YiLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUsIGhlbHA9IlN1Ym1pc3Npb24gQ1NWIHBhdGguIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tb3V0LWRpciIsIHR5cGU9UGF0aCwgcmVxdWlyZWQ9VHJ1ZSwgaGVscD0iT3V0cHV0IGRpcmVjdG9yeSBmb3IgLmdlZmYgZmlsZXMuIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tbm8tb3ZlcndyaXRlIiwgYWN0aW9uPSJzdG9yZV90cnVlIiwgaGVscD0iRG8gbm90IG92ZXJ3cml0ZSBleGlzdGluZyBnZWZmcy4iKQogICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKCkKCiAgICBjc3ZfdG9fZ2VmZnMoYXJncy5jc3YsIGFyZ3Mub3V0X2Rpciwgb3ZlcndyaXRlPW5vdCBhcmdzLm5vX292ZXJ3cml0ZSkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigp"
with open('csv_to_geffs.py', 'wb') as _f:
    _f.write(base64.b64decode(_B64))
print('csv_to_geffs.py written to working dir.')

In [ ]:
#!/usr/bin/env python3
"""
FSOT KAGGLE COMPETITIVE SUBMISSION
-----------------------------------
Port of the original program: C:\\Users\\damia\\Desktop\\fsot_rna_trinary_evolution_sim

Pipeline (matches kaggle_prototype_fsot_tracker + fsot_full_pipeline_test):
  zarr video → vision detect → FSOT SequenceTracker (fsot_core math) → submission.csv

Engines (BIOHUB_ENGINE):
  fsot       — ORIGINAL PROGRAM: peaks/cellpose detect + FSOT linking (DEFAULT)
  fsot_unet  — U-Net vision gateway + FSOT linking (competition detector upgrade)
  unet       — legacy cellmot transformer edges (not FSOT-first)
  cellpose   — alias for fsot + cellpose detector
  peaks      — alias for fsot + peaks detector

FSOT_LINK_MODE (fsot_unet only): fsot | hybrid | transformer
"""

from __future__ import annotations

import glob
import os
import sys
import zipfile
from pathlib import Path

import pandas as pd

LEAN_VERIFICATION_REPO = "https://github.com/dappalumbo91/FSOT-2.1-Lean.git"


def _program_root() -> Path:
    """Notebook-safe root (Jupyter has no __file__)."""
    if os.path.exists("/kaggle/working"):
        return Path("/kaggle/working")
    try:
        return Path(__file__).resolve().parent
    except NameError:
        return Path.cwd()


PROGRAM_ROOT = _program_root()

if os.path.exists("/kaggle/input"):
    print("[ENV] Kaggle")
    _test_dirs = glob.glob("/kaggle/input/**/test", recursive=True)
    DATA_DIR = _test_dirs[0] if _test_dirs else "/kaggle/input"
    OUT_CSV = "/kaggle/working/submission.csv"
    _ft = glob.glob("/kaggle/input/**/cellmot-ft-detector-biohub/**/edge_predictor_best.pth", recursive=True)
    _wt = _ft or glob.glob("/kaggle/input/**/edge_predictor_best.pth", recursive=True)
    if _wt:
        os.environ.setdefault("CELLMOT_UNET_WEIGHTS", _wt[0])
        print(f"[UNET] weights: {_wt[0]}")
    _cp = glob.glob("/kaggle/input/**/cpsam_v2", recursive=True)
    if _cp:
        os.environ.setdefault("CELLPOSE_WEIGHTS", _cp[0])

    def _extract_cellmot_bundle() -> None:
        candidates = glob.glob("/kaggle/input/**/cellmot_code_bundle.zip", recursive=True)
        if os.path.exists("/kaggle/working/cellmot_code_bundle.zip"):
            candidates.insert(0, "/kaggle/working/cellmot_code_bundle.zip")
        for bundle_path in candidates:
            with zipfile.ZipFile(bundle_path, "r") as zf:
                zf.extractall("/kaggle/working")
            sys.path.insert(0, "/kaggle/working/cellmot_bundle/src")
            sys.path.insert(0, "/kaggle/working/cellmot_bundle/scripts")
            print(f"[CELLMOT] bundle extracted from {bundle_path}")
            return
        print("[WARN] cellmot_code_bundle.zip not found — fsot_unet engine unavailable")

    _extract_cellmot_bundle()
    os.environ.setdefault("BIOHUB_ENGINE", "fsot_unet")
    os.environ.setdefault("BIOHUB_DETECTOR", "peaks")
    os.environ.setdefault("FSOT_LINK_MODE", "fsot_gate")
    os.environ.setdefault("FSOT_GATE_FRAC", "0.48")
    os.environ.setdefault("FSOT_GATE_ADAPTIVE", "0")
    os.environ.setdefault("FSOT_GATE_RESCUE", "0")
    os.environ.setdefault("CELLMOT_USE_FT", "1")
    os.environ.setdefault("CELLMOT_DET_THRESHOLD", "0.99")
    os.environ.setdefault("CELLMOT_EDGE_THRESHOLD", "0.3")
    os.environ.setdefault("CELLMOT_USE_ILP", "1")
    os.environ.setdefault("CELLMOT_ILP_MAX_EDGES", "35000")
    os.environ.setdefault("CELLMOT_DET_TTA", "0")
    os.environ.setdefault("CELLMOT_DEVICE", "cpu")
    os.environ.setdefault("CELLMOT_NMS_UM", "8.0")
    os.environ.setdefault("CELLMOT_POOL_UM", "8.0")
    os.environ.setdefault("FSOT_GAP_LINK", "1")
    os.environ.setdefault("KAGGLE_SUBMISSION_FAST_VALIDATE", "1")
else:
    print("[ENV] Local")
    DATA_DIR = r"D:\Kaggle_Biohub_Data\test"
    OUT_CSV = str(PROGRAM_ROOT / "submission_master.csv")
    _repo = PROGRAM_ROOT / "kaggle-cell-tracking-competition"
    if _repo.exists():
        sys.path.insert(0, str(_repo / "src"))
        sys.path.insert(0, str(_repo / "scripts"))

ENGINE = os.environ.get("BIOHUB_ENGINE", "auto").lower()


def _engine_available(name: str) -> bool:
    if name in ("fsot_unet", "unet"):
        try:
            from biohub_unet_engine import _resolve_weights
            _resolve_weights()
            return True
        except Exception:
            return False
    if name == "cellpose":
        try:
            import cellpose  # noqa: F401
            return True
        except Exception:
            return False
    return name in ("fsot", "peaks")


def _pick_engine() -> str:
    if ENGINE != "auto":
        return ENGINE
    # On Kaggle with U-Net weights, default to fsot_unet (matches notebook env).
    if os.path.exists("/kaggle/input") and _engine_available("fsot_unet"):
        return "fsot_unet"
    if os.environ.get("FSOT_PREFER_UNET", "0") == "1" and _engine_available("fsot_unet"):
        return "fsot_unet"
    return "fsot"


def _validate_submission_rows(rows: list[dict]) -> None:
    """Ensure node_id / edge refs reset per dataset (competition sample format)."""
    if os.environ.get("KAGGLE_SUBMISSION_FAST_VALIDATE", "0") != "1":
        import tempfile
        from pathlib import Path

        from biohub_unet_engine import write_submission_csv

        with tempfile.TemporaryDirectory(prefix="biohub_validate_") as tmp:
            csv_path = Path(tmp) / "submission.csv"
            write_submission_csv(rows, csv_path)
            try:
                from validate_kaggle_submission import validate_csv
            except ImportError:
                validate_csv = None
            if validate_csv is not None:
                errors = validate_csv(csv_path, strict_datasets=False)
                if errors:
                    raise ValueError("submission validation failed:\n  " + "\n  ".join(errors))
                return
    df = pd.DataFrame(rows)
    nodes = df[df["row_type"] == "node"]
    edges = df[df["row_type"] == "edge"]
    for ds in nodes["dataset"].unique():
        nd = nodes[nodes["dataset"] == ds]
        ed = edges[edges["dataset"] == ds]
        if int(nd["node_id"].min()) != 1:
            raise ValueError(f"{ds}: node_id must start at 1, got {nd['node_id'].min()}")
        if int(nd["node_id"].max()) != len(nd):
            raise ValueError(f"{ds}: node_id must be contiguous 1..N")
        node_set = set(nd["node_id"].tolist())
        bad = ed[(~ed["source_id"].isin(node_set)) | (~ed["target_id"].isin(node_set))]
        if len(bad):
            raise ValueError(f"{ds}: {len(bad)} edges reference invalid node_id values")


def _run_fsot_unet(data_dir: str, ml_edges: bool) -> list[dict]:
    """U-Net vision gateway (fsot_unet_gateway.py pattern) + FSOT linking."""
    from biohub_unet_engine import graph_to_submission_rows_from_graph, predict_graph

    if ml_edges:
        os.environ.setdefault("FSOT_LINK_MODE", "transformer")
        tag = "UNET-ML"
    else:
        os.environ.setdefault("FSOT_LINK_MODE", "fsot_gate")
        tag = "FSOT-GATE+UNET"

    rows: list[dict] = []
    row_idx = 0
    for ds_name in sorted(d for d in os.listdir(data_dir) if d.endswith(".zarr")):
        clean = ds_name.replace(".zarr", "")
        print(f"[{tag}] {clean}")
        graph = predict_graph(os.path.join(data_dir, clean))
        part, row_idx, _ = graph_to_submission_rows_from_graph(
            graph, clean, row_start=row_idx, node_id_start=1,
        )
        rows.extend(part)
    return rows


def _run_original_fsot(data_dir: str, detector: str) -> list[dict]:
    """Original program port: zarr → detect → FSOT SequenceTracker → rows."""
    from fsot_original_competition import track_all_datasets

    os.environ["BIOHUB_DETECTOR"] = detector
    return track_all_datasets(data_dir, detector_mode=detector)


def main():
    engine = _pick_engine()
    print("=" * 70)
    print("FSOT KAGGLE SUBMISSION — original program port")
    print(f"Program  : {PROGRAM_ROOT}")
    print(f"Lean ref : {LEAN_VERIFICATION_REPO}")
    print(f"Engine   : {engine}")
    print(f"Data dir : {DATA_DIR}")
    print("=" * 70)

    if not os.path.exists(DATA_DIR):
        print(f"FATAL: missing {DATA_DIR}")
        return

    if engine == "fsot_unet":
        rows = _run_fsot_unet(DATA_DIR, ml_edges=False)
    elif engine == "unet":
        rows = _run_fsot_unet(DATA_DIR, ml_edges=True)
    elif engine == "cellpose":
        rows = _run_original_fsot(DATA_DIR, "cellpose")
    elif engine == "peaks":
        rows = _run_original_fsot(DATA_DIR, "peaks")
    else:
        detector = os.environ.get("BIOHUB_DETECTOR", "peaks")
        rows = _run_original_fsot(DATA_DIR, detector)

    _validate_submission_rows(rows)
    from biohub_unet_engine import write_submission_csv

    write_submission_csv(rows, OUT_CSV)
    print(f"\n[COMPLETE] {len(rows)} rows -> {OUT_CSV}")
    print("=" * 70)


if __name__ == "__main__":
    main()